# Reinforcement Learning — Energy-Saving 5G/6G HetNet with Handover Analysis
### MSc Thesis: Energy Saving Operation of 5G and 6G Networks using AI
**Author:** Praneshraj Tiruppur Nagarajan Dhyaneswar  
**Supervisor:** Dr. Tibor Cinkler  
**Date:** June 2026

| Cell | Content |
|------|---------|
| 1 | Installs |
| 2 | Imports & Seeds |
| 3 | Radio parameters (SSL-aligned) |
| 4 | HetNet topology factory |
| 5 | Physics helpers (path loss, channel gain, SINR, EARTH power) |
| 6 | TRUST helpers |
| 7 | UE Mobility (Random Waypoint, 3 mobility classes) |
| 8 | Enhanced HandoverState (A3+TTT, ping-pong, CIT) |
| 9 | HetNetEnergyEnv (Gymnasium, BS warm-up, multi-objective reward) |
| 10 | ActorCritic network (per-BS encoder + shared trunk) |
| 11 | PPO update (GAE-λ, entropy annealing, clipped surrogate) |
| 12 | Training loop (3-seed protocol) |
| 13 | Baseline policies |
| 14 | Evaluation suite |
| 15 | Topology preview |
| 16 | BS activation heatmap |
| 17 | BS ON/OFF map snapshots |
| 18 | Training curves |
| 19 | UE trajectories + HO events |
| 20 | Bar charts vs baselines |
| 21 | HO-Energy Pareto frontier |
| 22 | Ablation study |
| 23 | Ablation radar chart |


## Cell 1 — Installs
Install `gymnasium` (OpenAI Gym successor) for the RL environment interface.

In [12]:
# Cell 1 — Installs
import subprocess, sys                                   # subprocess to call pip programmatically
for pkg in ['gymnasium']:                                # list of required packages not in standard colab
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])  # silent install
print('Packages ready.')


Packages ready.


## Cell 2 — Imports & Seeds
Standard scientific Python stack plus PyTorch, Gymnasium, and tqdm.  
Global seed fixed at 42 for reproducibility across all three libraries.

In [13]:
# Cell 2 — Imports and Seeds
import os                                                # file and directory operations
import math                                              # math utilities (ceil, log, etc.)
import random                                            # Python built-in RNG
import time                                              # timing utilities
import copy                                              # deep copy of objects
import warnings                                          # suppress noisy deprecation warnings
from dataclasses import dataclass, field                 # typed data containers for metrics

import numpy as np                                       # numerical arrays and linear algebra
import pandas as pd                                      # tabular data and evaluation DataFrames
import matplotlib.pyplot as plt                          # plotting and visualisation
import matplotlib.patches as mpatches                    # legend patch handles
from matplotlib.colors import ListedColormap             # custom colour maps for heatmaps

import torch                                             # PyTorch deep learning backend
import torch.nn as nn                                    # neural network modules
import torch.nn.functional as F                          # functional API (MSE loss, etc.)

import gymnasium as gym                                  # Gymnasium RL environment interface
from gymnasium import spaces                             # action and observation space classes
from tqdm.auto import tqdm                               # progress bars for training loops

warnings.filterwarnings('ignore')                        # suppress non-critical warnings

SEED = 42                                                # global reproducibility seed
random.seed(SEED)                                        # seed Python RNG
np.random.seed(SEED)                                     # seed NumPy RNG
torch.manual_seed(SEED)                                  # seed PyTorch CPU operations
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)                     # seed all CUDA devices if available

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # select compute device
print(f'Device: {DEVICE}')


Device: cpu


## Cell 3 — Radio Parameters (SSL-aligned)

### References
- **EARTH power model**: Auer, G. et al. (2011). *How Much Energy Is Needed to Run a Wireless Network?* IEEE Wireless Communications, 18(5), 40–49.  
- **3GPP path loss / band parameters**: 3GPP TR 38.901 V17.0.0 (2022). *Study on channel model for frequencies from 0.5 to 100 GHz*.  
- **HetNet tier definitions**: Lopez-Perez, D. et al. (2011). *OFDMA Femtocells: A Roadmap on Interference Avoidance.* IEEE Communications Magazine.  
- **SINR threshold / noise figure**: Andrews, J. G. et al. (2011). *Femtocells: Past, Present, and Future.* IEEE JSAC.  
- **Handover margin / TTT (A3 event)**: 3GPP TS 36.331 V17.3.0 (2022). *Radio Resource Control (RRC) — A3 event configuration.*  

All arrays are indexed by cell type: `CT_MACRO=0, CT_MICRO=1, CT_PICO=2, CT_FEMTO=3`.  
Parameters are identical to those in the SSL (Graph-SCP) notebook to ensure consistent physics.


In [14]:
# Cell 3 — Radio Parameters (aligned with SSL / Graph-SCP notebook)

# ── Cell-type integer constants ──────────────────────────────────────────────
CT_MACRO, CT_MICRO, CT_PICO, CT_FEMTO = 0, 1, 2, 3     # cell-type indices used throughout
CELLTYPE_NAMES = np.array(['Macro', 'Micro', 'Pico', 'Femto'], dtype=object)  # human-readable labels

# ── Frequency bands ───────────────────────────────────────────────────────────
BANDS_HZ = np.array([700e6, 3.5e9, 28e9], dtype=np.float64)     # carrier frequencies [Hz]: sub-1GHz, mid-band, mmWave
BAND_BW_HZ = np.array([20e6, 60e6, 400e6], dtype=np.float64)    # channel bandwidths [Hz] per band
CT_BAND = np.array([0, 1, 2, 2], dtype=np.int32)                 # cell-type → band index (Pico and Femto share mmWave)

# ── Antenna configuration (3GPP TR 38.901) ────────────────────────────────────
N_TX = np.array([8, 64, 256], dtype=np.int32)                    # transmit antenna elements per band (massive MIMO)
N_RX = 2                                                          # UE receive chains (diversity)
BF_GAIN_DB = 10.0 * np.log10(N_TX)                               # beamforming gain [dBi] = 10*log10(N_TX)
BS_GAIN_DBI = np.array([8., 8., 5.], dtype=np.float64)           # BS antenna gain [dBi] per band
UE_GAIN_DBI = 0.0                                                 # UE isotropic antenna gain [dBi]

# ── Noise model ────────────────────────────────────────────────────────────────
UE_NF_DB = 9.0                                                    # UE noise figure [dB] — typical LTE/NR value
NOISE_DBM = -174.0 + 10.0 * np.log10(BAND_BW_HZ) + UE_NF_DB    # thermal noise floor [dBm] per band
NOISE_W = 10.0 ** (NOISE_DBM / 10.0 - 3.0)                      # thermal noise [W] converted from dBm
NOISE_W_RX = NOISE_W * N_RX                                       # receiver-combined noise [W] (N_RX chains)

# ── SINR thresholds ────────────────────────────────────────────────────────────
SINR_THRESH_DB = -6.0                                             # minimum decodable SINR [dB] (QPSK-1/6 MCS)
SINR_CAP_DB = 40.0                                                # SINR cap [dB] to prevent log overflow

# ── EARTH BS power model (Auer et al. 2011) ───────────────────────────────────
# P_BS(ρ) = P0 + ΔP * P_TX * ρ  (active),   P_sleep  (sleep)
P0_W = np.array([354.0, 56.0, 6.8, 4.8], dtype=np.float64)      # baseline active power [W] at zero load
DELTA_P = np.array([4.7, 2.6, 4.0, 8.0], dtype=np.float64)      # load-dependent slope ΔP [dimensionless]
PTX_W = np.array([40.0, 6.3, 0.25, 0.10], dtype=np.float64)     # maximum RF transmit power [W]
PMAX_W = P0_W + DELTA_P * PTX_W                                   # maximum site power [W] at 100 % load
PSLEEP_W = np.array([17.7, 2.8, 0.34, 0.24], dtype=np.float64)  # sleep-mode power [W]

# ── Coverage geometry ─────────────────────────────────────────────────────────
MAX_DIST_M = np.array([2000., 500., 200., 50.], dtype=np.float64)  # hard coverage radius [m] per cell type
SHADOW_STD_DB = np.array([4., 6., 8., 10.], dtype=np.float64)     # log-normal shadowing σ [dB] per cell type

# ── Capacity limits ────────────────────────────────────────────────────────────
MAX_USERS_PER_BS = np.array([200, 80, 30, 10], dtype=np.int32)   # max UEs schedulable per BS per slot

# ── Diurnal traffic model (12-hour window, 10:00–21:00) ───────────────────────
HOUR_USERCOUNTS = {                                                # UE count per hour (busy-hour profile)
    10: 420, 11: 520, 12: 650, 13: 750, 14: 600, 15: 550,
    16: 680, 17: 820, 18: 900, 19: 850, 20: 680, 21: 450,
}
SLOT_HOURS = sorted(HOUR_USERCOUNTS.keys())                       # sorted list of hour labels
USERCOUNTS = [HOUR_USERCOUNTS[h] for h in SLOT_HOURS]            # ordered UE count per slot
N_SLOTS = len(USERCOUNTS)                                         # total number of time slots (12)

# ── Simulation area ────────────────────────────────────────────────────────────
AREA_M = 800.0                                                    # square area side length [m]
MIN_BS_GAP_M = 100.0                                              # minimum inter-BS spacing [m]
TARGET_COVERAGE = 0.995                                           # target UE coverage fraction (100 %)
MAX_USERS = max(USERCOUNTS)                                       # peak UE count for array sizing

# # ── PPO hyper-parameters (Schulman et al. 2017) ───────────────────────────────
# PPO_HIDDEN = 256                                                   #
# PPO_LR = 0.001                                                   #
# PPO_TOTAL_STEPS = 30_000                                         #
# PPO_ROLLOUT_LEN = 2048                                            #
# PPO_CLIP_EPS = 0.2                                                #
# PPO_VF_COEF = 0.5                                                 #
# PPO_ENT_START = 0.01                                              #
# PPO_ENT_END = 0.001                                               #
# PPO_N_EPOCHS = 4                                                  #
# PPO_BATCH_SIZE = 256                                               # mini-batch size for SGD
# PPO_GAMMA = 0.99                                                  #
# PPO_LAM = 0.95                                                    #
# PPO_MAX_GRAD = 0.5                                                #


PPO_HIDDEN = 256                                  # hidden layer width in ActorCritic
PPO_LR = 0.0005                                   # Slightly lower LR to prevent collapse with smaller batches, Adam learning rate
PPO_TOTAL_STEPS = 20_000                          # total environment interaction steps
PPO_ROLLOUT_LEN = 1000                            # DECREASE to 1000. This gives 20 total updates. rollout buffer length before update
PPO_CLIP_EPS = 0.2                                # PPO clipping epsilon ε
PPO_VF_COEF = 0.5                                 # value function loss coefficient
PPO_ENT_START = 0.05                              # entropy bonus coefficient (start)
PPO_ENT_END = 0.005                               # entropy bonus coefficient (end, annealed)
PPO_N_EPOCHS = 10                                 # INCREASE epochs (from 4 to 10) to squeeze more learning out of the short rollouts
PPO_BATCH_SIZE = 125                              # DECREASE batch size (so 1000/125 = 8 mini-batches per epoch)
PPO_GAMMA = 0.99                                  # discount factor γ
PPO_LAM = 0.95                                    # GAE-λ advantage estimation parameter
PPO_MAX_GRAD = 0.5                                # gradient clipping norm threshold

# ── Multi-objective reward weights ────────────────────────────────────────────
W_ENERGY = 1.0                                                    # weight for energy efficiency reward term
W_COV = 10.0                                                      # weight for coverage reward term
LAMBDA_COV = 50.0                                                 # penalty multiplier for coverage deficit,# MASSIVE increase
W_HO = 2.0                                                        # weight for handover rate penalty
W_PING_PONG = 2.0                                                 # weight for ping-pong HO penalty
W_CIT = 1.0                                                       # weight for cell-in-trouble (CIT) penalty




# ── TRUST score parameters (Graph-SCP alignment) ─────────────────────────────
TRUST_ALPHA = 0.5                                                 # TRUST: channel quality weight
TRUST_GAMMA = 0.3                                                 # TRUST: load complement weight

# ── Handover parameters (3GPP TS 36.331 A3 event) ────────────────────────────
HO_MARGIN_DB = 3.0                                                # A3 hysteresis margin [dB]
HO_TTT = 3                                                        # Time-to-Trigger: slots before HO executes
PING_PONG_WINDOW = 3                                              # slots within which a back-HO counts as ping-pong
T_ON_SLOTS = 1                                                    # BS warm-up delay slots after OFF→ON

OUTPUT_DIR = 'output/GraphSCPHetNetRL'                            # directory for saved plots and CSV files
os.makedirs(OUTPUT_DIR, exist_ok=True)                            # create directory if it does not exist

print('P0_W      :', P0_W)
print('DELTA_P   :', DELTA_P)
print('USERCOUNTS:', USERCOUNTS)
print('TARGET_COV:', TARGET_COVERAGE)


P0_W      : [354.   56.    6.8   4.8]
DELTA_P   : [4.7 2.6 4.  8. ]
USERCOUNTS: [420, 520, 650, 750, 600, 550, 680, 820, 900, 850, 680, 450]
TARGET_COV: 0.995


## Cell 4 — HetNet Topology Factory

### References
- **4-tier HetNet model**: Hoydis, J., Brink, S., & Debbah, M. (2013). *Massive MIMO in the UL/DL of Cellular Networks: How Many Antennas Do We Need?* IEEE JSAC, 31(2), 160–171.  
- **Maximin placement**: Andrews, J. G., Baccelli, F., & Ganti, R. K. (2011). *A Tractable Approach to Coverage and Rate in Cellular Networks.* IEEE Trans. Communications.  
- **Deployment ratios**: ETSI TR 136 872 V12.1.0 (2015). *Evolved Universal Terrestrial Radio Access (E-UTRA); Dual Connectivity.*


In [15]:
# Cell 4 — HetNet topology factory (4-tier, 21 BSs)

# BS count ratios per scenario size
BAND_RATIO = {
    21: {'macro': 3, 'micro': 6, 'pico': 8, 'femto': 4},   # default 21-BS scenario
    25: {'macro': 3, 'micro': 7, 'pico': 10, 'femto': 5},  # extended 25-BS scenario
}


def maximin_place(n_place, fixed_pts, area_m=800., min_gap=100., seed=42):
    """Place n_place BSs using a maximin strategy: maximise the minimum distance to already-placed BSs."""
    rng = np.random.default_rng(seed)                            # local RNG for reproducibility
    candidates = []                                              # candidate grid points
    for r in np.arange(80., area_m / 2., 20.):                 # radial sweep from centre outward
        for a in np.arange(0., 360., 5.):                      # angular sweep every 5 degrees
            x = float(np.clip(area_m / 2 + r * np.cos(np.radians(a)), 40., area_m - 40.))
            y = float(np.clip(area_m / 2 + r * np.sin(np.radians(a)), 40., area_m - 40.))
            candidates.append((x, y))
    rng.shuffle(candidates)                                      # randomise order to break ties
    placed = list(map(tuple, fixed_pts)) if fixed_pts is not None else []  # already-placed BSs
    new_pts = []                                                 # newly placed BS coordinates
    for _ in range(n_place):
        best_pt, best_score = None, -1.                         # track best candidate this iteration
        for cand in candidates:
            if cand in placed:                                   # skip already-placed locations
                continue
            dists = [np.hypot(cand[0] - p[0], cand[1] - p[1]) for p in placed]
            if placed and min(dists) < min_gap:                  # enforce minimum gap constraint
                continue
            score = min(dists) if dists else 1e9                # score = min distance (maximised)
            if score > best_score:
                best_pt, best_score = cand, score
        if best_pt is None:
            raise RuntimeError(f'Cannot place {n_place} BSs with gap {min_gap} m')
        placed.append(best_pt)
        new_pts.append(best_pt)
    return np.array(new_pts, dtype=np.float64)


def make_hetnet(n_bs=21, area_m=800., seed=42):
    """Build a 4-tier HetNet topology and return a DataFrame + coordinate/type arrays."""
    counts = BAND_RATIO.get(n_bs)                               # fetch tier counts for this scenario
    if counts is None:
        raise ValueError(f'Unsupported n_bs={n_bs}')
    n_macro = counts['macro']                                    # number of macro BSs
    n_micro = counts['micro']                                    # number of micro BSs
    n_pico = counts['pico']                                      # number of pico BSs
    n_femto = counts['femto']                                    # number of femto BSs

    cx, cy = area_m / 2., area_m / 2.                           # area centre coordinates
    # Place macros on a 290 m radius ring (tri-sector layout)
    macro_xy = np.array([
        (cx + 290. * np.cos(np.radians(a)), cy + 290. * np.sin(np.radians(a)))
        for a in [90., 210., 330.][:n_macro]
    ], dtype=np.float64)

    rows, idx = [], 0                                            # row accumulator and global BS index
    for k, (x, y) in enumerate(macro_xy, 1):
        rows.append({'bs_id': idx, 'label': f'Macro-{k:02d}', 'xm': float(x), 'ym': float(y), 'ct': CT_MACRO})
        idx += 1

    micro_xy = maximin_place(n_micro, macro_xy, min_gap=150., seed=seed + 11)  # place micros away from macros
    for k, (x, y) in enumerate(micro_xy, 1):
        rows.append({'bs_id': idx, 'label': f'Micro-{k:02d}', 'xm': float(x), 'ym': float(y), 'ct': CT_MICRO})
        idx += 1

    # Place picos and femtos together (share mmWave band), then split
    small_xy = maximin_place(n_pico + n_femto, np.vstack([macro_xy, micro_xy]), min_gap=80., seed=seed + 29)
    for k, (x, y) in enumerate(small_xy[:n_pico], 1):
        rows.append({'bs_id': idx, 'label': f'Pico-{k:02d}', 'xm': float(x), 'ym': float(y), 'ct': CT_PICO})
        idx += 1
    for k, (x, y) in enumerate(small_xy[n_pico:], 1):
        rows.append({'bs_id': idx, 'label': f'Femto-{k:02d}', 'xm': float(x), 'ym': float(y), 'ct': CT_FEMTO})
        idx += 1

    bs_df = pd.DataFrame(rows)                                   # build BS metadata DataFrame
    bs_xy = bs_df[['xm', 'ym']].values.astype(np.float64)       # (n_bs, 2) coordinate array
    bs_ct = bs_df['ct'].values.astype(np.int32)                  # (n_bs,) cell-type integer array
    print(f'HetNet {n_bs} BSs  macro={np.sum(bs_ct == CT_MACRO)} '
          f'micro={np.sum(bs_ct == CT_MICRO)} pico={np.sum(bs_ct == CT_PICO)} femto={np.sum(bs_ct == CT_FEMTO)}')
    return bs_df, bs_xy, bs_ct


NUM_BS = 21                                                       # active scenario: 21-BS network
bs_df, BS_XY, BS_CT = make_hetnet(NUM_BS)                        # build topology
MAX_NET_POWER_W = float(np.sum(PMAX_W[BS_CT]))                   # theoretical maximum power [W] (all BSs at 100 % load)

# Visual style maps per cell type
CT_COLORS  = {CT_MACRO: '#1565C0', CT_MICRO: '#2E7D32', CT_PICO: '#FBC02D', CT_FEMTO: '#E65100'}
CT_MARKERS = {CT_MACRO: '^', CT_MICRO: 's', CT_PICO: 'D', CT_FEMTO: 'o'}
CT_SIZES   = {CT_MACRO: 200, CT_MICRO: 120, CT_PICO: 80, CT_FEMTO: 60}

print(f'Max network power: {MAX_NET_POWER_W:.1f} W')
print(bs_df[['label', 'ct', 'xm', 'ym']].to_string(index=False))


HetNet 21 BSs  macro=3 micro=6 pico=8 femto=4
Max network power: 2145.1 W
   label  ct         xm         ym
Macro-01   0 400.000000 690.000000
Macro-02   0 148.852633 255.000000
Macro-03   0 651.147367 255.000000
Micro-01   1 729.089653 590.000000
Micro-02   1  70.910347 590.000000
Micro-03   1 400.000000  40.000000
Micro-04   1 400.000000 320.000000
Micro-05   1 277.432889 502.846018
Micro-06   1 522.567111 502.846018
 Pico-01   2 590.000000 729.089653
 Pico-02   2 210.000000 729.089653
 Pico-03   2 210.000000  70.910347
 Pico-04   2 590.000000  70.910347
 Pico-05   2 760.000000 400.000000
 Pico-06   2  40.000000 400.000000
 Pico-07   2 492.976018 200.612287
 Pico-08   2 307.023982 200.612287
Femto-01   3 554.548132 358.588953
Femto-02   3 245.451868 358.588953
Femto-03   3 400.000000 560.000000
Femto-04   3 148.859285 467.292952


## Cell 5 — Physics Helpers (Path Loss, Channel Gain, SINR, EARTH Power)

### References
- **3GPP dual-slope path loss**: 3GPP TR 38.901 V17.0.0 (2022). Equations (7.4.1-1) and (7.4.2-1).  
- **Log-normal shadowing**: Goldsmith, A. (2005). *Wireless Communications.* Cambridge University Press. Chapter 2.  
- **EARTH linear BS power model**: Auer, G. et al. (2011). *How Much Energy Is Needed to Run a Wireless Network?* IEEE Wireless Communications, 18(5), 40–49.  
- **Greedy UE association**: Andrews, J. G. et al. (2013). *What Will 5G Be?* IEEE JSAC, 32(6), 1065–1082.


In [16]:
# Cell 5 — Wireless physics helpers (3GPP TR 38.901 + EARTH power model)


def pathloss_db(dist_m, fc_hz, band_idx):
    """3GPP TR 38.901 dual-slope path loss model [dB]."""
    fc_ghz = fc_hz / 1e9                                         # carrier frequency in GHz
    d = np.maximum(dist_m, 1.)                                   # clip minimum distance to 1 m
    h_bs = 25. if band_idx == 0 else 10.                         # BS height [m]: macro=25, small cell=10
    h_ut = 1.5                                                    # UE height [m] (standard pedestrian)
    d_bp = 4. * (h_bs - 1.) * (h_ut - 1.) * fc_hz / 3e8         # breakpoint distance [m]

    if band_idx == 0:                                             # sub-1 GHz (700 MHz) macro model
        pl1 = 28. + 22. * np.log10(d) + 20. * np.log10(fc_ghz)
        pl2 = 28. + 40. * np.log10(d) + 20. * np.log10(fc_ghz) - 9. * np.log10(d_bp**2 + (h_bs - h_ut)**2)
    else:                                                         # mid-band / mmWave UMi model
        pl1 = 32.4 + 21. * np.log10(d) + 20. * np.log10(fc_ghz)
        pl2 = 32.4 + 40. * np.log10(d) + 20. * np.log10(fc_ghz) - 9.5 * np.log10(d_bp**2 + (h_bs - h_ut)**2)
    return np.where(d < d_bp, pl1, pl2)                          # piecewise: near-field vs far-field slope


def channel_gain_linear(ue_xy, bs_xy, bs_ct, rng=None):
    """Vectorised: no Python loop over BSs."""
    n_ue, n_bs = ue_xy.shape[0], bs_xy.shape[0]
    bands = CT_BAND[bs_ct]                                       # (n_bs,)
    fc_hz = BANDS_HZ[bands]                                      # (n_bs,)
    fc_ghz = fc_hz / 1e9

    # distances: (n_ue, n_bs)
    dx = ue_xy[:, 0:1] - bs_xy[:, 0]
    dy = ue_xy[:, 1:2] - bs_xy[:, 1]
    dist = np.maximum(np.sqrt(dx**2 + dy**2), 1.)

    # per-BS height and breakpoint distance (broadcast over n_ue)
    h_bs = np.where(bands == 0, 25., 10.)                        # (n_bs,)
    d_bp = 4. * (h_bs - 1.) * 0.5 * fc_hz / 3e8                # (n_bs,)

    # dual-slope path loss (vectorised piecewise)
    macro = (bands == 0)                                         # (n_bs,)
    pl1 = np.where(macro,
                   28. + 22.*np.log10(dist) + 20.*np.log10(fc_ghz),
                   32.4 + 21.*np.log10(dist) + 20.*np.log10(fc_ghz))
    pl2 = np.where(macro,
                   28. + 40.*np.log10(dist) + 20.*np.log10(fc_ghz)
                       - 9. *np.log10(d_bp**2 + (h_bs - 1.5)**2),
                   32.4 + 40.*np.log10(dist) + 20.*np.log10(fc_ghz)
                       - 9.5*np.log10(d_bp**2 + (h_bs - 1.5)**2))
    pl = np.where(dist < d_bp, pl1, pl2)                        # (n_ue, n_bs)

    if rng is not None:
        pl += rng.normal(0., SHADOW_STD_DB[bs_ct], size=(n_ue, n_bs))

    gain_db = BS_GAIN_DBI[bands] + UE_GAIN_DBI + BF_GAIN_DB[bands] - pl
    cg = 10. ** (gain_db / 10.)
    cg[dist > MAX_DIST_M[bs_ct]] = 0.
    return cg


def sinr_db_matrix(cg, activemask, bs_ct):
    """Compute SINR [dB] for every UE-BS pair, accounting for co-channel interference."""
    n_ue, n_bs = cg.shape
    pt = PTX_W[bs_ct]                                            # transmit power [W] per BS
    prx = cg * pt[None, :]                                       # received power [W]: P_rx = G * P_tx
    out = np.full_like(prx, -120.)                               # initialise SINR matrix to noise floor
    mask = activemask.astype(float)                              # float mask for active BSs

    for j in range(n_bs):
        b = CT_BAND[bs_ct[j]]                                    # band index of BS j
        noise = float(NOISE_W_RX[b])                             # thermal noise [W] for this band
        mj = mask.copy()
        mj[j] = 0.                                               # exclude BS j from its own interference sum
        interf = np.sum(prx * mj[None, :], axis=1)              # total co-channel interference [W]
        denom = np.maximum(interf + noise, 1e-20)                # avoid division by zero
        ratio = np.maximum(prx[:, j], 1e-20) / denom            # SINR linear ratio
        out[:, j] = np.minimum(10. * np.log10(ratio), SINR_CAP_DB)  # convert to dB and cap

    return out


def greedy_association(cg, activemask, bs_ct, sinr_mat=None):
    """Assign each UE to its best feasible BS (highest SINR), respecting capacity limits."""
    n_ue, n_bs = cg.shape
    if sinr_mat is None:
        sinr_mat = sinr_db_matrix(cg, activemask, bs_ct)         # compute SINR matrix if not provided
    feas = (sinr_mat >= SINR_THRESH_DB) & (activemask[None, :] == 1)  # feasible UE-BS links

    loads = np.zeros(n_bs, dtype=np.int32)                       # current load (UE count) per BS
    served = np.zeros(n_ue, dtype=bool)                          # served flag per UE
    serving = -np.ones(n_ue, dtype=np.int32)                     # serving BS index (-1 = unserved)
    best_sinr = np.where(feas, sinr_mat, -np.inf).max(axis=1)   # best feasible SINR per UE

    for i in np.argsort(-best_sinr):                             # process UEs from best-to-worst SINR
        for j in np.argsort(-np.where(feas[i], sinr_mat[i], -np.inf)):  # try BSs in decreasing SINR order
            if not feas[i, j]:
                break                                             # no more feasible BSs for this UE
            cap = MAX_USERS_PER_BS[bs_ct[j]]                     # capacity of BS j
            if loads[j] < cap:                                   # accept if BS has room
                loads[j] += 1
                served[i] = True
                serving[i] = j
                break

    load_ratio = loads.astype(float) / MAX_USERS_PER_BS[bs_ct]  # normalised load ρ ∈ [0,1]
    return served, serving, loads, load_ratio, sinr_mat


def network_power_w(activemask, load_ratio, bs_ct):
    """EARTH linear power model: P_j = P0 + ΔP*PTX*ρ (on) or P_sleep (off)."""
    total = 0.0                                                   # accumulate total network power [W]
    for j, active in enumerate(activemask):
        ct = bs_ct[j]                                             # cell type of BS j
        if active:
            # EARTH model: baseline + load-proportional term
            total += P0_W[ct] + DELTA_P[ct] * PTX_W[ct] * float(np.clip(load_ratio[j], 0., 1.))
        else:
            total += PSLEEP_W[ct]                                 # sleep-mode consumption
    return float(total)


print('Physics helpers loaded.')


Physics helpers loaded.


## Cell 6 — TRUST Helpers (Voronoi Expected Load & Trust Score)

### References
- **TRUST score concept**: Derived from Graph-SCP (this thesis). Channel quality-to-load ratio used to rank BS reliability in the semi-supervised labelling pipeline.  
- **Voronoi-based load estimation**: Haenggi, M. (2012). *Stochastic Geometry for Wireless Networks.* Cambridge University Press.


In [17]:
# Cell 6 — TRUST helpers (Voronoi-based expected load and BS trust score)


def compute_feasibility(cg, bs_ct):
    """Compute binary feasibility matrix: can UE i be served by BS j when j is the only active BS?"""
    n_ue, n_bs = cg.shape
    feas = np.zeros((n_ue, n_bs), dtype=bool)                    # initialise all links as infeasible
    for j in range(n_bs):
        mj = np.zeros(n_bs, dtype=np.int32)
        mj[j] = 1                                                 # activate only BS j
        feas[:, j] = sinr_db_matrix(cg, mj, bs_ct)[:, j] >= SINR_THRESH_DB  # check per-link SINR
    return feas


def compute_expected_load(cg, feas, bs_ct):
    """Estimate expected load ρ̂ per BS using Voronoi-like best-channel assignment (no capacity limit)."""
    n_ue, n_bs = cg.shape
    counts = np.zeros(n_bs, dtype=np.float32)                    # soft UE count per BS
    for i in range(n_ue):
        js = np.where(feas[i])[0]                                 # feasible BSs for UE i
        if len(js):
            counts[js[np.argmax(cg[i, js])]] += 1.               # assign UE to BS with highest gain
    cap = np.maximum(MAX_USERS_PER_BS[bs_ct].astype(np.float32), 1.)  # per-BS capacity
    return np.clip(counts / cap, 0., 1.)                         # normalised expected load ρ̂ ∈ [0,1]


def compute_trust(cg, feas, rho_hat, bs_ct, alpha=0.5, gamma_t=0.3):
    """Compute normalised TRUST score: combination of channel quality uniformity and spare capacity."""
    n_ue, n_bs = cg.shape
    scores = np.zeros(n_bs, dtype=np.float32)                    # raw trust scores
    for j in range(n_bs):
        uid = np.where(feas[:, j])[0]                            # UEs feasibly covered by BS j
        if len(uid) == 0:
            continue                                              # skip uncovered BSs
        cg_j = cg[uid, j]                                        # channel gains to covered UEs
        # TRUST = α * (mean_gain / max_gain) + γ * (1 - load)
        scores[j] = (alpha * float(cg_j.mean() / (cg_j.max() + 1e-9))
                     + gamma_t * float(1. - rho_hat[j]))
    mx = scores.max()
    return (scores / (mx + 1e-9)).astype(np.float32)             # normalise to [0, 1]


print('TRUST helpers loaded.')


TRUST helpers loaded.


## Cell 7 — UE Mobility Model (Random Waypoint, 3 Mobility Classes)

### References
- **Random Waypoint model**: Bettstetter, C., Hartenstein, H., & Pérez-Costa, X. (2004). *Stochastic Properties of the Random Waypoint Mobility Model.* Wireless Networks, 10(5), 555–567.  
- **Mobility class mix (static/pedestrian/vehicular)**: 3GPP TR 38.913 V15.0.0 (2018). *Study on Scenarios and Requirements for Next Generation Access Technologies.* Table 7.1-1.


In [18]:
# Cell 7 — UE Mobility Model (Random Waypoint, 3 mobility classes)


class UEMobilityModel:
    """Simulate UE positions using the Random Waypoint (RWP) model.

    Three mobility classes are mixed:
      - static (20 %): office/indoor users, speed ≈ 0
      - pedestrian (60 %): 0.3–1.5 m/s
      - vehicular (20 %): 3–20 m/s
    """
    SPEED_RANGES = {
        'static': (0., 0.1),           # near-zero speed for static UEs
        'pedestrian': (0.3, 1.5),      # walking speed range [m/s]
        'vehicular': (3., 20.),        # vehicle speed range [m/s]
    }
    CLASSES = ['static', 'pedestrian', 'vehicular']  # mobility class labels
    CLASS_PROBS = [0.20, 0.60, 0.20]                 # class mixture probabilities (must sum to 1)

    def __init__(self, n_ue, area_m=800., rng=None):
        self.n_ue = n_ue                                          # number of UEs to simulate
        self.area_m = area_m                                      # simulation area side length [m]
        self.rng = rng or np.random.default_rng(SEED + 123)      # RNG with deterministic seed
        self.pos = self.rng.uniform(0., area_m, size=(n_ue, 2))  # initial random positions
        self.wp = self.rng.uniform(0., area_m, size=(n_ue, 2))   # initial random waypoints
        self.speed = np.zeros(n_ue, dtype=np.float32)            # speed [m/s] per UE
        self.cls = self.rng.choice(self.CLASSES, size=n_ue, p=self.CLASS_PROBS)  # assign mobility class
        self._resample(np.arange(n_ue))                          # draw initial speeds

    def _resample(self, idx):
        """Draw new random speeds for UEs that have reached their waypoint."""
        for i in idx:
            lo, hi = self.SPEED_RANGES[self.cls[i]]
            self.speed[i] = self.rng.uniform(lo, hi)             # uniform speed within class range

    def step(self, dt=1.):
        """Advance UE positions by one time step dt [s] towards their current waypoints."""
        vec = self.wp - self.pos                                  # direction vector to waypoint
        dist = np.linalg.norm(vec, axis=1) + 1e-9               # distance to waypoint (avoid /0)
        dirn = vec / dist[:, None]                                # unit direction vector
        self.pos += (self.speed * dt)[:, None] * dirn            # move towards waypoint

        arrived = dist <= self.speed * dt                        # UEs that reached their waypoint
        if np.any(arrived):
            self.wp[arrived] = self.rng.uniform(0., self.area_m, size=(arrived.sum(), 2))  # new waypoint
            self._resample(np.where(arrived)[0])                  # draw new speed for arrived UEs

        self.pos = np.clip(self.pos, 0., self.area_m)            # keep UEs within area boundary
        return self.pos.copy()


print('UEMobilityModel loaded (RWP, 3 classes).')


UEMobilityModel loaded (RWP, 3 classes).


## Cell 8 — Enhanced HandoverState (A3+TTT, Ping-Pong, CIT)

### References
- **A3 event + Time-to-Trigger**: 3GPP TS 36.331 V17.3.0 (2022). Section 5.5.4.4 — *Event A3: Neighbour becomes offset better than SpCell.*  
- **Ping-pong handover definition**: Dimou, K. et al. (2009). *Handover within 3GPP LTE: Design Principles and Performance.* IEEE VTC Fall, 1–5.  
- **Cell-in-Trouble (CIT)**: Aliu, O. G. et al. (2013). *A Survey of Self Organisation in Future Cellular Networks.* IEEE Communications Surveys & Tutorials, 15(1), 336–361.  
- **Policy-driven vs mobility-driven HO classification**: Novel contribution — classifies HOs triggered by RL BS switch-off versus natural UE movement.


In [19]:
# Cell 8 — Enhanced HandoverState
# Tracks: A3+TTT trigger, ping-pong detection, Cell-in-Trouble (CIT), policy vs mobility HO split


class HandoverState:
    """Track per-UE handover events across time slots.

    Attributes tracked:
      - serving: current serving BS index per UE
      - prev_serving / prev_prev_serving: history for ping-pong detection
      - ttt: Time-to-Trigger counter matrix (n_ue × n_bs)
      - cit_slots: cumulative slots each UE spent without a serving BS (Cell-in-Trouble)
      - last_ho_slot: slot of most recent HO per UE (for ping-pong window)
      - prev_activemask: previous slot's BS active mask (to classify policy-driven HOs)
    """

    def __init__(self, n_ue, n_bs):
        self.n_ue = n_ue                                          # number of UEs
        self.n_bs = n_bs                                          # number of BSs
        self.serving = -np.ones(n_ue, dtype=np.int32)            # serving BS (-1 = unserved)
        self.prev_serving = -np.ones(n_ue, dtype=np.int32)       # previous serving BS
        self.prev_prev_serving = -np.ones(n_ue, dtype=np.int32)  # two slots ago (ping-pong check)
        self.ttt = np.zeros((n_ue, n_bs), dtype=np.int32)        # A3 TTT counter per UE-BS pair
        self.cit_slots = np.zeros(n_ue, dtype=np.int32)          # cumulative CIT slots per UE
        self.last_ho_slot = -np.ones(n_ue, dtype=np.int32)       # slot of last HO (-1 = never)
        self.slot = 0                                             # current slot index
        self.prev_activemask = np.ones(n_bs, dtype=np.int32)     # previous slot's active BS mask

    def update(self, sinr_mat, activemask):
        nue = sinr_mat.shape[0]
        self.slot += 1

        # ── CIT: vectorised ──────────────────────────────────────────────
        s_arr = self.serving[:nue]
        safe_s = np.maximum(s_arr, 0)
        cit_mask = (s_arr < 0) | (activemask[safe_s] == 0)
        self.cit_slots[:nue] += cit_mask.astype(np.int32)

        # ── Best BS per UE (vectorised argmax) ───────────────────────────
        active_sinr = np.where(activemask[None, :] == 1, sinr_mat, -np.inf)
        bj_arr = np.argmax(active_sinr, axis=1)               # (nue,)
        sinr_b = sinr_mat[np.arange(nue), bj_arr]
        sinr_s = np.where(s_arr >= 0, sinr_mat[np.arange(nue), safe_s], -120.)

        # ── A3 trigger / TTT ─────────────────────────────────────────────
        a3 = sinr_b > sinr_s + HO_MARGIN_DB
        # increment TTT for triggered (best BS, per UE) — still a small indexed write
        triggered_ue = np.where(a3)[0]
        self.ttt[:nue][triggered_ue, bj_arr[triggered_ue]] += 1
        self.ttt[:nue][~a3] = 0                               # reset non-triggered rows

        # ── HO execution (vectorised) ────────────────────────────────────
        ttt_ready = self.ttt[:nue][np.arange(nue), bj_arr] >= HO_TTT
        target_active = activemask[bj_arr] == 1
        ho_ok = ttt_ready & target_active
        ho_fail = ttt_ready & ~target_active

        # Ping-pong mask
        pp_mask = (ho_ok
                  & (self.prev_prev_serving[:nue] == bj_arr)
                  & (self.last_ho_slot[:nue] >= 0)
                  & (self.slot - self.last_ho_slot[:nue] <= PING_PONG_WINDOW))

        # Policy-driven mask
        pol_mask = (ho_ok
                    & (s_arr >= 0)
                    & (self.prev_activemask[safe_s] == 1)
                    & (activemask[safe_s] == 0))

        # Commit HOs
        self.prev_prev_serving[:nue][ho_ok] = self.prev_serving[:nue][ho_ok]
        self.prev_serving[:nue][ho_ok] = s_arr[ho_ok]
        self.serving[:nue][ho_ok] = bj_arr[ho_ok]
        self.last_ho_slot[:nue][ho_ok] = self.slot
        self.ttt[:nue][ho_ok] = 0
        self.ttt[:nue][ho_fail] = 0

        self.prev_activemask = activemask.astype(np.int32)

        n_ho = int(ho_ok.sum())
        return dict(
            nhandovers=n_ho,
            nhosuccess=n_ho,
            nhofailure=int(ho_fail.sum()),
            nhopingpong=int(pp_mask.sum()),
            nhopolicy=int(pol_mask.sum()),
            nhomobility=n_ho - int(pol_mask.sum()),
        )


print('HandoverState loaded.')
print('Tracks: A3+TTT, ping-pong, CIT, policy vs mobility split.')


HandoverState loaded.
Tracks: A3+TTT, ping-pong, CIT, policy vs mobility split.


## Cell 9 — HetNetEnergyEnv (Gymnasium RL Environment)

**Observation space** (214-dim vector):  
- 21 × 10 per-BS features: `[active, load, norm_SINR, norm_power, trust, cell_type_norm, ho_rate, fail_rate, pp_rate, policy_ho_rate]`  
- 4 global features: `[coverage, slot_fraction, ue_fraction, global_ho_rate]`

**Action space**: `MultiBinary(21)` — desired ON/OFF state per BS.

**Reward**:  
`R = W_E·r_E + W_C·r_C + W_H·r_H + W_PP·r_PP + W_CI·r_CI`

where:
- `r_E = -P_total / P_max` — energy efficiency
- `r_C = +1` if coverage ≥ target, else `-λ_cov · deficit` — coverage satisfaction
- `r_H = -HO_rate / n_UE` — handover rate penalty
- `r_PP = -ping_pong_rate` — ping-pong penalty
- `r_CI = -CIT_total / (n_UE × slot)` — cell-in-trouble penalty

### References
- **Gymnasium API**: Towers, M. et al. (2024). *Gymnasium: A Standard Interface for Reinforcement Learning Environments.* arXiv:2407.17032.  
- **Multi-objective reward shaping**: Sutton, R. S. & Barto, A. G. (2018). *Reinforcement Learning: An Introduction.* 2nd ed. MIT Press. Chapter 17.  
- **BS warm-up / state machine**: 3GPP TR 36.902 V9.3.1 (2011). *Self-configuring and self-optimizing network (SON) use cases.* Section 5.4.


In [20]:
# Cell 9 — HetNetEnergyEnv (Gymnasium, BS warm-up state machine, multi-objective reward)
# Obs: 21×10 per-BS features + 4 global = 214-dim
# Act: MultiBinary(21) — desired ON/OFF for each BS
# Rew: W_E*r_E + W_C*r_C + W_H*r_H + W_PP*r_PP + W_CI*r_CI


class HetNetEnergyEnv(gym.Env):
    """Gymnasium environment for energy-efficient BS activation in a 4-tier HetNet."""

    metadata = {'render.modes': []}
    BS_OFF = 0                                                    # BS state: powered down (sleep)
    BS_WARMING = 1                                                # BS state: warming up (transitioning ON)
    BS_ON = 2                                                     # BS state: fully active

    def __init__(self, bs_xy, bs_ct, uecounts, area_m=800., nslots=12,
                 use_trust=True, t_on=1, w_energy=1., w_cov=10., lambda_cov=20.,
                 w_ho=0.5, w_pp=2., w_ci=1., traffic_type='random'):
        super().__init__()
        self.bs_xy = np.asarray(bs_xy, dtype=np.float64)         # BS coordinates (n_bs, 2)
        self.bs_ct = np.asarray(bs_ct, dtype=np.int32)           # BS cell types (n_bs,)
        self.n_bs = len(bs_ct)                                    # total number of BSs
        self.uecounts = list(uecounts)                            # UE count per slot (traffic schedule)
        self.nslots = nslots                                      # episode length in slots
        self.area_m = area_m                                      # simulation area [m]
        self.use_trust = use_trust                                # whether to compute TRUST features
        self.t_on = max(int(t_on), 0)                            # warm-up delay [slots] after OFF→ON
        self.traffic_type = traffic_type                          # 'random' or 'hotspot'

        # Reward weights
        self.w_energy = w_energy                                  # energy reward weight
        self.w_cov = w_cov                                        # coverage reward weight
        self.lambda_cov = lambda_cov                              # coverage deficit penalty multiplier
        self.w_ho = w_ho                                          # handover rate penalty weight
        self.w_pp = w_pp                                          # ping-pong penalty weight
        self.w_ci = w_ci                                          # cell-in-trouble penalty weight

        self.n_bs_feats = 10                                      # per-BS feature dimension
        obs_dim = self.n_bs * self.n_bs_feats + 4                # total observation dimension
        self.action_space = spaces.MultiBinary(self.n_bs)        # binary ON/OFF per BS
        self.observation_space = spaces.Box(-1., 1., shape=(obs_dim,), dtype=np.float32)
        self.max_power = float(np.sum(PMAX_W[self.bs_ct]))       # theoretical peak power [W]
        self.rng = np.random.default_rng(SEED + 999)             # environment-specific RNG

        # Episode state variables (initialised in reset)
        self.slot = 0
        self.ue_xy = None                                         # current UE positions
        self.mob = None                                           # UE mobility model instance
        self.hostate = None                                       # handover state tracker
        self.activemask = None                                    # binary active-BS mask
        self.bs_state = np.full(self.n_bs, self.BS_ON, dtype=np.int32)   # per-BS state machine
        self.bs_timer = np.zeros(self.n_bs, dtype=np.int32)      # warm-up countdown per BS
        self.coverage = 0.                                        # fraction of served UEs
        self.ptotal = 0.                                          # current network power [W]
        self.history = []                                         # step-by-step log for visualisation

    def _apply_sm(self, desired_on):
        """Apply the BS state machine: OFF→WARMING→ON and ON→OFF transitions."""
        desired_on = desired_on.astype(np.int32)
        for j in range(self.n_bs):
            s = self.bs_state[j]
            if s == self.BS_ON:
                if desired_on[j] == 0:                           # agent wants to turn off an active BS
                    self.bs_state[j] = self.BS_OFF
                    self.bs_timer[j] = 0
            elif s == self.BS_OFF:
                if desired_on[j] == 1:                           # agent wants to turn on a sleeping BS
                    if self.t_on > 0:
                        self.bs_state[j] = self.BS_WARMING       # must warm up first
                        self.bs_timer[j] = self.t_on
                    else:
                        self.bs_state[j] = self.BS_ON            # instant on (no warm-up)
            elif s == self.BS_WARMING:
                self.bs_timer[j] -= 1                            # count down warm-up timer
                if self.bs_timer[j] <= 0:
                    self.bs_state[j] = self.BS_ON                # warm-up complete → active
                    self.bs_timer[j] = 0

        am = (self.bs_state == self.BS_ON).astype(np.int32)      # binary active mask
        if am.sum() == 0:                                         # safety: at least one macro must be on
            j0 = np.where(self.bs_ct == CT_MACRO)[0][0]
            self.bs_state[j0] = self.BS_ON
            am[j0] = 1
        return am

    def _build_obs(self, am, serving, lr, sinr_mat, n_ue, trust_n, ev):
        """Build the flattened observation vector from current environment state."""
        pn = float(np.clip(self.ptotal / self.max_power, 0., 1.))  # normalised total power [0,1]
        n_ho = ev.get('nhandovers', 0)                           # total HOs this slot
        n_pp = ev.get('nhopingpong', 0)                          # ping-pong HOs this slot
        n_fail = ev.get('nhofailure', 0)                         # failed HOs this slot
        n_pol = ev.get('nho_policy', 0)                          # policy-driven HOs this slot

        bs_feats = []
        for j in range(self.n_bs):
            uj = np.where(serving == j)[0]                       # UEs currently served by BS j
            mean_sinr = float(sinr_mat[uj, j].mean()) if len(uj) > 0 else -120.  # mean SINR for served UEs
            # Normalise SINR to [-1, 1] relative to [SINR_THRESH_DB, SINR_CAP_DB]
            sinr_norm = float(np.clip(
                (mean_sinr - SINR_THRESH_DB) / (SINR_CAP_DB - SINR_THRESH_DB) * 2. - 1., -1., 1.))
            load_val = float(lr[j])                              # normalised load ρ ∈ [0,1]
            load_safe = max(load_val, 1e-3)                      # avoid division by zero in rate features
            ct_norm = float(self.bs_ct[j]) / 3.                 # normalised cell type [0=Macro … 1=Femto]
            trust_val = float(trust_n[j]) if trust_n is not None else 0.  # TRUST score ∈ [0,1]
            ho_rate_feat = float(n_ho / max(n_ue, 1)) * load_safe        # scaled HO rate
            fail_rate_feat = float(n_fail / max(n_ho, 1)) * load_safe if n_ho > 0 else 0.  # HO failure rate
            pp_rate_feat = float(n_pp / max(n_ho, 1)) * load_safe if n_ho > 0 else 0.     # ping-pong rate
            pol_rate_feat = float(n_pol / max(n_ho, 1)) * load_safe if n_ho > 0 else 0.   # policy-HO rate

            # 10 features per BS
            bs_feats += [float(am[j]), load_val, sinr_norm, pn,
                         trust_val, ct_norm, ho_rate_feat, fail_rate_feat, pp_rate_feat, pol_rate_feat]

        # 4 global features
        global_feats = [
            float(self.coverage),                                 # current coverage fraction
            float(self.slot) / float(self.nslots),               # episode progress [0,1]
            float(n_ue) / float(max(self.uecounts)),             # relative UE load [0,1]
            float(np.clip(n_ho / max(n_ue, 1), 0., 1.)),        # global HO rate per UE
        ]
        return np.clip(np.array(bs_feats + global_feats, dtype=np.float32), -1., 1.)

    def _sample_ue(self, n_ue):
        """Sample UE initial positions: uniform random or clustered hotspot."""
        if self.traffic_type == 'hotspot':
            n_clusters = int(self.rng.integers(5, 9))            # 5–8 random hotspot centres
            centres = self.rng.uniform(.1 * self.area_m, .9 * self.area_m, size=(n_clusters, 2))
            assign = self.rng.choice(n_clusters, size=n_ue)      # assign each UE to a cluster
            pos = np.zeros((n_ue, 2))
            for i in range(n_ue):
                pos[i] = centres[assign[i]] + self.rng.normal(0., 35., size=2)  # Gaussian spread ±35 m
            return np.clip(pos, 0., self.area_m)
        return self.rng.uniform(0., self.area_m, size=(n_ue, 2))  # uniform random (IID)

    def reset(self, seed=None, options=None):
        """Reset the environment to the start of a new episode."""
        if seed is not None:
            self.rng = np.random.default_rng(seed)

        # Random start slot so the agent sees all hours during training
        self.slot = int(self.rng.integers(0, self.nslots))

        n_ue = self.uecounts[self.slot]                           # UE count for starting slot

        # FIX 1: Initialize with MAX_USERS so the arrays never need to be destroyed/recreated
        self.mob = UEMobilityModel(MAX_USERS, area_m=self.area_m, rng=self.rng)
        self.mob.pos = self._sample_ue(MAX_USERS)                # initial UE positions for MAX_USERS

        # Only take the positions for the currently active UEs
        self.ue_xy = self.mob.pos[:n_ue].copy()

        # FIX 1 (cont): Initialize handover state for MAX_USERS
        self.hostate = HandoverState(MAX_USERS, self.n_bs)       # fresh handover tracker

        self.bs_state[:] = self.BS_ON                             # all BSs start active
        self.bs_timer[:] = 0
        self.activemask = np.ones(self.n_bs, dtype=np.int32)     # all BSs on at episode start
        self.history = []

        # Compute initial channel and association
        cg = channel_gain_linear(self.ue_xy, self.bs_xy, self.bs_ct, self.rng)
        served, serving, loads, lr, sinr_mat = greedy_association(cg, self.activemask, self.bs_ct)
        feas = compute_feasibility(cg, self.bs_ct)
        rho_hat = compute_expected_load(cg, feas, self.bs_ct)
        trust_n = compute_trust(cg, feas, rho_hat, self.bs_ct) if self.use_trust else None

        self.coverage = float(served.mean())
        self.ptotal = network_power_w(self.activemask, lr, self.bs_ct)

        # Pass only the active UEs' SINR to HandoverState to prevent index errors
        ev = self.hostate.update(sinr_mat, self.activemask)

        self.history.append({
            'slot': self.slot, 'bs_state': self.bs_state.copy(),
            'activemask': self.activemask.copy(), 'ue_xy': self.ue_xy.copy(),
            'serving': serving.copy(), 'coverage': self.coverage,
            'power': self.ptotal, 'hoevents': ev,
        })
        obs = self._build_obs(self.activemask, serving, lr, sinr_mat, n_ue, trust_n, ev)
        info = dict(coverage=self.coverage, powerw=self.ptotal,
                    nactivebs=int(self.activemask.sum()), nue=n_ue, hoevents=ev,
                    r_E=0., r_C=0., r_H=0., r_PP=0., r_CI=0.)
        return obs, info

    def step(self, action):
        """Advance the environment by one time slot given the agent's BS activation action."""
        self.activemask = self._apply_sm(np.asarray(action, dtype=np.int32))  # apply state machine
        self.slot += 1
        n_ue = self.uecounts[min(self.slot, self.nslots - 1)]    # UE count for this slot

        # Removed the `if self.mob is None or self.mob.n_ue != n_ue:` check.
        # Since self.mob handles MAX_USERS, it never needs to be recreated.

        # FIX 2: Fast inner loop to simulate physical seconds while the RL step remains 1 hour
        FAST_SIM_STEPS = 60
        FAST_DT = 1.0

        # Accumulate handover events across the 60 physical seconds
        total_ev = {'nhandovers': 0, 'nhopingpong': 0, 'nhosuccess': 0,
                    'nhofailure': 0, 'nhopolicy': 0, 'nhomobility': 0}

        for _ in range(FAST_SIM_STEPS):
            # Advance all UEs in the pool by 1 physical second
            full_ue_xy = self.mob.step(dt=FAST_DT)

            # Slice only the currently active UEs for this hour
            self.ue_xy = full_ue_xy[:n_ue].copy()

            # Recompute channel and association for active UEs
            cg = channel_gain_linear(self.ue_xy, self.bs_xy, self.bs_ct, self.rng)
            served, serving, loads, lr, sinr_mat = greedy_association(cg, self.activemask, self.bs_ct)

            # Update handover state and get events for this 1-second step
            step_ev = self.hostate.update(sinr_mat, self.activemask)

            for k in total_ev:
                total_ev[k] += step_ev.get(k, 0)

        # Recompute final TRUST after the physical loop
        feas = compute_feasibility(cg, self.bs_ct)
        rho_hat = compute_expected_load(cg, feas, self.bs_ct)
        trust_n = compute_trust(cg, feas, rho_hat, self.bs_ct) if self.use_trust else None

        self.coverage = float(served.mean())
        self.ptotal = network_power_w(self.activemask, lr, self.bs_ct)
        ev = total_ev  # Replace the single event dict with the accumulated one

        n_ho = ev.get('nhandovers', 0)
        n_pp = ev.get('nhopingpong', 0)

        # Scale CIT down by the fast steps so the penalty remains equivalent to the original scale
        total_cit = float(self.hostate.cit_slots[:n_ue].sum()) / FAST_SIM_STEPS

        # ── Multi-objective reward components ─────────────────────────────────
        r_E = -self.ptotal / self.max_power                       # energy: negative normalised power
        cov_deficit = max(0., TARGET_COVERAGE - self.coverage)   # shortfall below target coverage
        r_C = 1. if cov_deficit == 0. else -self.lambda_cov * cov_deficit  # +1 if met, else large penalty
        r_H = -(n_ho / max(n_ue, 1))                             # handover rate penalty
        r_PP = -(n_pp / max(n_ho, 1)) if n_ho > 0 else 0.       # ping-pong fraction penalty
        r_CI = -(total_cit / max(float(n_ue * self.slot), 1.))  # cumulative CIT penalty

        reward = (self.w_energy * r_E + self.w_cov * r_C
                  + self.w_ho * r_H + self.w_pp * r_PP + self.w_ci * r_CI)  # weighted sum

        terminated = (self.slot >= self.nslots - 1)              # episode ends after N_SLOTS

        self.history.append({
            'slot': self.slot, 'bs_state': self.bs_state.copy(),
            'activemask': self.activemask.copy(), 'ue_xy': self.ue_xy.copy(),
            'serving': serving.copy(), 'coverage': self.coverage,
            'power': self.ptotal, 'hoevents': ev,
        })
        obs = self._build_obs(self.activemask, serving, lr, sinr_mat, n_ue, trust_n, ev)
        info = dict(coverage=self.coverage, powerw=self.ptotal,
                    nactivebs=int(self.activemask.sum()), nue=n_ue, hoevents=ev,
                    r_E=r_E, r_C=r_C, r_H=r_H, r_PP=r_PP, r_CI=r_CI)
        return obs, float(reward), terminated, False, info


# Instantiate environment to verify dimensions
env = HetNetEnergyEnv(BS_XY, BS_CT, USERCOUNTS, use_trust=True, t_on=T_ON_SLOTS)
obs0, info0 = env.reset()
print('Env OK  obs_dim:', env.observation_space.shape[0], ' n_bs:', env.n_bs)
print(f'Max power: {env.max_power:.1f} W')


Env OK  obs_dim: 214  n_bs: 21
Max power: 2145.1 W


## Cell 10 — ActorCritic Network (per-BS Encoder + Shared Trunk)

Architecture: each BS's feature vector is encoded independently by a shared 2-layer MLP,
then all encodings are concatenated with global features and passed through a shared trunk.
The trunk feeds into separate actor (logit) and critic (value) heads.

### References
- **PPO actor-critic architecture**: Schulman, J. et al. (2017). *Proximal Policy Optimization Algorithms.* arXiv:1707.06347.  
- **Shared encoder for multi-agent / multi-entity RL**: Wen, M. et al. (2022). *Multi-Agent Reinforcement Learning is a Sequence Modeling Problem.* NeurIPS 2022.  
- **Bernoulli action distribution for binary actions**: Berner, C. et al. (2019). *Dota 2 with Large Scale Deep Reinforcement Learning.* arXiv:1912.06680.


In [21]:

# Cell 10 — PPO Actor-Critic: per-BS encoder (shared weights) + global trunk


class ActorCritic(nn.Module):
    """Actor-critic network for MultiBinary(n_bs) action space.

    Per-BS features are encoded by a shared 2-layer MLP (weight-tied across BSs),
    then concatenated with global features and processed by a shared trunk.
    Separate linear heads produce per-BS logits (actor) and a scalar value (critic).
    """

    def __init__(self, n_bs, bs_feats, n_global=4, hidden=256):
        super().__init__()
        self.n_bs = n_bs                                          # number of BSs
        self.bs_feats = bs_feats                                  # features per BS

        # Per-BS encoder: maps bs_feats → 64 (shared weights across all BSs)
        self.bs_enc = nn.Sequential(
            nn.Linear(bs_feats, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
        )
        trunk_in = n_bs * 64 + n_global                          # encoder output + global features
        # Shared trunk: fused representation for both actor and critic
        self.trunk = nn.Sequential(
            nn.Linear(trunk_in, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.actor = nn.Linear(hidden, n_bs)                     # per-BS activation logits
        self.critic = nn.Linear(hidden, 1)                       # scalar state-value estimate

    def _encode(self, x):
        """Encode observation: split per-BS and global features, encode BSs, concatenate."""
        bd = self.n_bs * self.bs_feats                           # length of per-BS feature block
        # Reshape per-BS block to (batch, n_bs, bs_feats) and encode with shared weights
        h = self.bs_enc(x[:, :bd].view(-1, self.n_bs, self.bs_feats))  # (batch, n_bs, 64)
        return self.trunk(torch.cat([h.reshape(-1, self.n_bs * 64), x[:, bd:]], dim=-1))

    def forward(self, x):
        h = self._encode(x)
        return self.actor(h), self.critic(h).squeeze(-1)         # logits (n_bs,), value scalar

    @torch.no_grad()
    def act(self, obs_np, deterministic=False):
        """Sample or greedily select an action from the current policy."""
        obs_t = torch.as_tensor(obs_np, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        logits, val = self.forward(obs_t)
        dist = torch.distributions.Bernoulli(logits=logits)      # independent Bernoulli per BS
        action = dist.probs.gt(.5).float() if deterministic else dist.sample()  # deterministic=argmax
        logp = dist.log_prob(action).sum(-1)                     # sum log-probs across BSs
        return action.squeeze(0).cpu().numpy().astype(np.int32), float(logp.item()), float(val.item())

    def evaluate(self, obs_t, act_t):
        """Evaluate log-prob, entropy, and value for a batch of (obs, action) pairs."""
        logits, val = self.forward(obs_t)
        dist = torch.distributions.Bernoulli(logits=logits)
        return dist.log_prob(act_t).sum(-1), dist.entropy().sum(-1), val


model = ActorCritic(n_bs=env.n_bs, bs_feats=env.n_bs_feats).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=PPO_LR)
print(f'ActorCritic trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')


ActorCritic trainable params: 421,654


## Cell 11 — PPO Update (GAE-λ, Entropy Annealing, Clipped Surrogate)

### References
- **PPO clipped surrogate objective**: Schulman, J. et al. (2017). *Proximal Policy Optimization Algorithms.* arXiv:1707.06347. Eq. (7).  
- **Generalised Advantage Estimation (GAE)**: Schulman, J. et al. (2016). *High-Dimensional Continuous Control Using Generalized Advantage Estimation.* ICLR 2016.  
- **Entropy regularisation for exploration**: Williams, R. J. & Peng, J. (1991). *Function Optimization using Connectionist Reinforcement Learning Algorithms.* Connection Science, 3(3), 241–268.


In [22]:
# Cell 11 — PPO update (GAE-λ advantage, entropy annealing, clipped surrogate loss)


def ppo_update(model, optimizer, rollout, global_step, total_steps):
    """Perform one PPO update cycle over the collected rollout buffer."""
    # ── Unpack rollout ─────────────────────────────────────────────────────────
    obs_list, act_list, logp_list, rew_list, val_list, done_list = zip(*rollout)
    obs_t = torch.as_tensor(np.stack(obs_list), dtype=torch.float32, device=DEVICE)   # (T, obs_dim)
    act_t = torch.as_tensor(np.stack(act_list), dtype=torch.float32, device=DEVICE)   # (T, n_bs)
    logp_old = torch.as_tensor(np.array(logp_list), dtype=torch.float32, device=DEVICE)  # (T,)
    rews = np.array(rew_list, dtype=np.float32)                  # (T,) rewards
    vals = np.array(val_list, dtype=np.float32)                  # (T,) critic values
    dones = np.array(done_list, dtype=np.float32)                # (T,) episode termination flags

    # ── GAE-λ advantage computation (Schulman et al. 2016) ───────────────────
    adv = np.zeros_like(rews)
    gae = 0.
    for t in reversed(range(len(rews))):
        nxt_val = 0. if (t == len(rews) - 1 or dones[t]) else vals[t + 1]  # bootstrap next value
        delta = rews[t] + PPO_GAMMA * nxt_val * (1 - dones[t]) - vals[t]   # TD error δ
        gae = delta + PPO_GAMMA * PPO_LAM * (1 - dones[t]) * gae            # recursive GAE
        adv[t] = gae
    returns = adv + vals                                         # targets for value function

    adv_t = torch.as_tensor(adv, dtype=torch.float32, device=DEVICE)
    ret_t = torch.as_tensor(returns, dtype=torch.float32, device=DEVICE)
    adv_t = (adv_t - adv_t.mean()) / (adv_t.std() + 1e-8)      # normalise advantages (stabilises training)

    # ── Linearly anneal entropy coefficient ───────────────────────────────────
    frac = min(float(global_step) / float(max(total_steps, 1)), 1.)  # training progress [0,1]
    ent_coef = PPO_ENT_START + frac * (PPO_ENT_END - PPO_ENT_START)  # decreasing entropy bonus

    # ── PPO mini-batch update epochs ──────────────────────────────────────────
    n = len(obs_t)
    idx = np.arange(n)
    loss_val = 0.
    for _ in range(PPO_N_EPOCHS):
        np.random.shuffle(idx)                                   # shuffle for mini-batch sampling
        for s in range(0, n, PPO_BATCH_SIZE):
            b = idx[s: s + PPO_BATCH_SIZE]                      # mini-batch indices
            logp, ent, val = model.evaluate(obs_t[b], act_t[b]) # re-evaluate policy on mini-batch

            ratio = torch.exp(logp - logp_old[b])               # importance-sampling ratio π/π_old
            # Clipped surrogate: min of unclipped and clipped objectives (PPO Eq. 7)
            surr1 = ratio * adv_t[b]
            surr2 = torch.clamp(ratio, 1. - PPO_CLIP_EPS, 1. + PPO_CLIP_EPS) * adv_t[b]
            actor_loss = -torch.min(surr1, surr2).mean()        # maximise clipped surrogate
            critic_loss = F.mse_loss(val, ret_t[b])             # MSE value function loss
            loss = actor_loss + PPO_VF_COEF * critic_loss - ent_coef * ent.mean()  # combined loss

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), PPO_MAX_GRAD)  # gradient clipping
            optimizer.step()
            loss_val = float(loss.item())

    return loss_val


print('PPO update ready.')


PPO update ready.


## Cell 12 — Training Loop

configured to train using seed 42. Assess variance across training runs,
following best practices for RL benchmarking (Agarwal et al. 2021).

### References
- **Reliable RL evaluation (3+ seeds)**: Agarwal, R. et al. (2021). *Deep Reinforcement Learning at the Edge of the Statistical Precipice.* NeurIPS 2021.  
- **On-policy rollout collection**: Schulman, J. et al. (2017). *Proximal Policy Optimization Algorithms.* arXiv:1707.06347. Section 3.


In [ ]:

# ── Updated PPO Parameters for 20k steps ──


TRAIN_SEEDS   = [42]
CKPT_PATH_FMT = os.path.join(OUTPUT_DIR, 'rl_ckpt_seed{}.pt')
SAVE_EVERY    = 500

all_results = {}

for run_seed in TRAIN_SEEDS:
    print(f'\n=== Training seed {run_seed} ===')
    ckpt_path = CKPT_PATH_FMT.format(run_seed)
    resume    = os.path.exists(ckpt_path)

    # Reset seeds globally per run
    torch.manual_seed(run_seed)
    np.random.seed(run_seed)
    random.seed(run_seed)

    run_model = ActorCritic(n_bs=env.n_bs, bs_feats=env.n_bs_feats).to(DEVICE)
    run_opt   = torch.optim.Adam(run_model.parameters(), lr=PPO_LR)
    run_env   = HetNetEnergyEnv(BS_XY, BS_CT, USERCOUNTS, use_trust=True, t_on=T_ON_SLOTS)

    logs = dict(rewards=[], cov=[], power=[], ho_rate=[],
                r_E=[], r_C=[], r_H=[], r_PP=[], r_CI=[])
    start_step = 0

    # Resume from checkpoint if it exists
    if resume:
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        run_model.load_state_dict(ckpt['model'])
        run_opt.load_state_dict(ckpt['optimizer'])
        logs       = ckpt['logs']
        start_step = ckpt['step']

        if 'env_slot' in ckpt:
          run_env.reset(seed=run_seed)           # initialise arrays first
          run_env.slot = ckpt['env_slot']
          run_env.bs_state[:] = ckpt['env_bs_state']
          run_env.activemask = ckpt['env_activemask']
          run_env.mob.pos[:] = ckpt['mob_pos']
          run_env.mob.wp[:] = ckpt['mob_wp']
          run_env.mob.speed[:] = ckpt['mob_speed']
          run_env.hostate.serving[:] = ckpt['ho_serving']
          run_env.hostate.ttt[:] = ckpt['ho_ttt']
          run_env.hostate.cit_slots[:] = ckpt['ho_cit']
          print(f'  Resumed from checkpoint at step {start_step}')

        # If this seed is already fully trained, skip the while loop
        if start_step >= PPO_TOTAL_STEPS:
            print(f'  Seed {run_seed} already complete. Skipping to next.')
            all_results[run_seed] = {'model': run_model, 'env': run_env, 'logs': logs}
            continue
    else:
        print(f'  Starting fresh.')

    obs, info = run_env.reset(seed=run_seed)
    rollout   = []

    progress = tqdm(total=PPO_TOTAL_STEPS, initial=start_step, desc=f'Seed {run_seed}')
    step_idx = start_step

    while step_idx < PPO_TOTAL_STEPS:
        action, logp, val = run_model.act(obs)
        nobs, reward, term, trunc, info = run_env.step(action)
        rollout.append((obs, action.astype(np.float32), logp, reward, val, float(term)))
        obs = nobs
        step_idx += 1
        progress.update(1)

        # Log per-step metrics
        logs['rewards'].append(reward)
        logs['cov'].append(info['coverage'])
        logs['power'].append(info['powerw'])
        n_ho = info['hoevents'].get('nhandovers', 0)
        logs['ho_rate'].append(n_ho / max(info['nue'], 1))
        for k in ['r_E', 'r_C', 'r_H', 'r_PP', 'r_CI']:
            logs[k].append(info[k])

        # Live metrics in tqdm every 500 steps
        if step_idx % 500 == 0 and len(logs['rewards']) >= 100:
            progress.set_description(
                f"Seed {run_seed} | "
                f"Rwd: {np.mean(logs['rewards'][-100:]):.2f} | "
                f"Cov: {np.mean(logs['cov'][-100:]):.2f} | "
                f"Pwr: {np.mean(logs['power'][-100:]):.0f}W"
            )

        # PPO update
        if len(rollout) >= PPO_ROLLOUT_LEN:
            ppo_update(run_model, run_opt, rollout, step_idx, PPO_TOTAL_STEPS)
            rollout = []

        if term:
            obs, info = run_env.reset()

        # Checkpoint
        if step_idx % SAVE_EVERY == 0:
            torch.save({
    'step': step_idx,
    'model': run_model.state_dict(),
    'optimizer': run_opt.state_dict(),
    'logs': logs,
    # Environment state for seamless resume
    'env_slot': run_env.slot,
    'env_bs_state': run_env.bs_state.copy(),
    'env_activemask': run_env.activemask.copy(),
    'mob_pos': run_env.mob.pos.copy(),
    'mob_wp': run_env.mob.wp.copy(),
    'mob_speed': run_env.mob.speed.copy(),
    'ho_serving': run_env.hostate.serving.copy(),
    'ho_ttt': run_env.hostate.ttt.copy(),
    'ho_cit': run_env.hostate.cit_slots.copy(),
}, ckpt_path)

    progress.close()

    # Final save and store
    torch.save({
    'step': step_idx,
    'model': run_model.state_dict(),
    'optimizer': run_opt.state_dict(),
    'logs': logs,
    # Environment state for seamless resume
    'env_slot': run_env.slot,
    'env_bs_state': run_env.bs_state.copy(),
    'env_activemask': run_env.activemask.copy(),
    'mob_pos': run_env.mob.pos.copy(),
    'mob_wp': run_env.mob.wp.copy(),
    'mob_speed': run_env.mob.speed.copy(),
    'ho_serving': run_env.hostate.serving.copy(),
    'ho_ttt': run_env.hostate.ttt.copy(),
    'ho_cit': run_env.hostate.cit_slots.copy(),
}, ckpt_path)

    print(f'  Mean reward  (last 200): {np.mean(logs["rewards"][-200:]):.3f}')
    print(f'  Mean coverage(last 200): {np.mean(logs["cov"][-200:]):.3f}')
    print(f'  Mean power   (last 200): {np.mean(logs["power"][-200:]):.1f} W')
    print(f'  Mean HO rate (last 200): {np.mean(logs["ho_rate"][-200:]):.4f}/UE')

    all_results[run_seed] = {'model': run_model, 'env': run_env, 'logs': logs}


# Compatibility for downstream cells
primary_seed = TRAIN_SEEDS[0]
trained_model = all_results[primary_seed]['model']
trained_model.eval()
print('\nTraining complete.')


=== Training seed 42 ===
  Starting fresh.


Seed 42:   0%|          | 0/20000 [00:00<?, ?it/s]

## Cell 13 — Baseline Policies

Four heuristic baselines for comparison against the PPO agent:

| Policy | Description |
|--------|-------------|
| Always-On | All 21 BSs permanently active (upper bound on power) |
| Random | Random binary action, macros always on |
| Threshold | Turn off BSs with load < 10 % for 2+ consecutive slots |
| Greedy-Energy | Activate BSs in decreasing TRUST score order until ≈coverage target |

### References
- **Load-based sleep heuristic (Threshold)**: Son, K. et al. (2011). *Base Station Operation and User Association Mechanisms for Energy-Delay Tradeoffs in Green Cellular Networks.* IEEE JSAC, 29(8), 1525–1536.  
- **Greedy coverage heuristic**: Soh, Y. S. et al. (2013). *Energy Efficient Heterogeneous Cellular Networks.* IEEE JSAC, 31(5), 840–850.


In [ ]:
import torch

# 1. Define the path where the checkpoint was saved
ckpt_path = f'{OUTPUT_DIR}/rl_ckpt_seed42.pt'

# 2. Re-instantiate the model architecture
# You need a dummy environment just to get the correct dimensions (n_bs, n_bs_feats)
temp_env = HetNetEnergyEnv(BS_XY, BS_CT, USERCOUNTS)
loaded_model = ActorCritic(n_bs=temp_env.n_bs, bs_feats=temp_env.n_bs_feats).to(DEVICE)

# 3. Load the checkpoint dictionary
checkpoint = torch.load(ckpt_path, map_location=DEVICE)

# 4. Inject the saved weights into the model architecture
loaded_model.load_state_dict(checkpoint['model'])

# 5. Set to evaluation mode (turns off dropout/batchnorm training behavior if present)
loaded_model.eval()

print(f"Model successfully loaded from step {checkpoint['step']}")

# Now you can use `loaded_model.act(obs, deterministic=True)` just like before!

In [ ]:
# Cell 13 — Baseline policies (heuristic comparators for RL evaluation)


class AlwaysOnPolicy:
    """All BSs permanently ON — maximum power, upper-bound baseline."""
    def act(self, obs_np, deterministic=True):
        return np.ones(env.n_bs, dtype=np.int32), 0., 0.         # action=all-ones, logp=0, val=0


class RandomPolicy:
    """Uniformly random ON/OFF action, with macro BSs always forced ON."""
    def __init__(self, n_bs, bs_ct, rng_seed=0):
        self.n_bs = n_bs
        self.bs_ct = bs_ct
        self.rng = np.random.default_rng(rng_seed)               # independent RNG for reproducibility

    def act(self, obs_np, deterministic=False):
        act = self.rng.integers(0, 2, size=self.n_bs).astype(np.int32)  # random binary action
        act[np.where(self.bs_ct == CT_MACRO)[0][0]] = 1          # macro must always stay on
        return act, 0., 0.


class ThresholdPolicy:
    """Turn off non-macro BSs that have been underloaded (< load_thresh) for >= patience slots."""
    def __init__(self, n_bs, bs_ct, load_thresh=0.10, patience=2):
        self.n_bs = n_bs
        self.bs_ct = bs_ct
        self.load_thresh = load_thresh                            # load threshold below which BS is idle
        self.patience = patience                                  # consecutive idle slots before switch-off
        self._idle = np.zeros(n_bs, dtype=np.int32)              # idle counter per BS
        self._state = np.ones(n_bs, dtype=np.int32)              # current ON/OFF state

    def act(self, obs_np, deterministic=True):
        act = self._state.copy()
        for j in range(self.n_bs):
            load_val = float(obs_np[j * 10 + 1])                 # load feature at index 1 per BS
            if load_val < self.load_thresh:
                self._idle[j] += 1                               # increment idle counter
            else:
                self._idle[j] = 0                                # reset if BS becomes loaded
            if self._idle[j] >= self.patience and self.bs_ct[j] != CT_MACRO:
                act[j] = 0                                       # switch off idle non-macro BS
            else:
                act[j] = max(act[j], 0)
        act[np.where(self.bs_ct == CT_MACRO)[0][0]] = 1          # macro always on
        self._state = act
        return act, 0., 0.


class GreedyEnergyPolicy:
    """Activate BSs in decreasing TRUST score order until approximate coverage target is met."""
    def __init__(self, n_bs, bs_ct):
        self.n_bs = n_bs
        self.bs_ct = bs_ct

    def act(self, obs_np, deterministic=True):
        trust_vals = np.array([obs_np[j * 10 + 4] for j in range(self.n_bs)])  # TRUST feature per BS
        order = np.argsort(-trust_vals)                          # descending TRUST order
        act = np.zeros(self.n_bs, dtype=np.int32)
        for j in range(self.n_bs):
            if self.bs_ct[j] == CT_MACRO:
                act[j] = 1                                       # macros always on
        for j in order:
            act[j] = 1                                           # activate next-most-trusted BS
            if min(act.sum() / self.n_bs * 1.3, 1.) >= TARGET_COVERAGE:
                break                                            # stop when proxy coverage estimate reached
        return act, 0., 0.


baseline_policies = {
    'Always-On': AlwaysOnPolicy(),
    'Random': RandomPolicy(env.n_bs, BS_CT, rng_seed=7),
    'Threshold': ThresholdPolicy(env.n_bs, BS_CT),
    'Greedy': GreedyEnergyPolicy(env.n_bs, BS_CT),
}
print('Baseline policies:', list(baseline_policies.keys()))


## Cell 14 — Evaluation Suite

`run_episode` rolls out a single episode deterministically and returns an `EpisodeMetrics` dataclass.  
`evaluate_all_policies` averages metrics over multiple episodes for all policies including the RL agent.


In [ ]:
# ### What changed and why (Reasoning):
# 1. **Timeframe & Step Context (The "Snippet")**: Added an explicit printout showing that each episode represents a **12-hour diurnal traffic cycle (10:00 to 21:00)**, totaling 12 RL steps per episode. By running 3 episodes, we are evaluating exactly **36 hours of simulated network time** per policy.
# 2. **Policy Logic Breakdown**: Added a clear print block that explains *how* the RL agent differs from the traditional approaches. RL is **proactive** (balancing multiple objectives), Threshold is **reactive** (waiting for low load), Greedy is **heuristic** (sorting by TRUST), and Always-On is the **baseline upper-bound**.
# 3. **Docstrings**: Upgraded the `EpisodeMetrics` and `run_episode` descriptions so the code acts as self-documenting material for your thesis.

# Cell 14 — Evaluation suite (episode rollout + policy comparison)

from dataclasses import dataclass
import pandas as pd

@dataclass
class EpisodeMetrics:
    """
    Container for per-episode evaluation metrics.
    In this thesis, an episode represents a 12-hour diurnal traffic cycle (10:00 to 21:00).
    Each step in the episode corresponds to 1 hour, containing 60 physical fast-sim seconds
    where UEs move and handover logic is evaluated.
    """
    mean_power: float = 0.          # mean network power [W]
    mean_cov: float = 0.            # mean coverage fraction
    energy_saving_pct: float = 0.   # energy saving relative to P_max [%]
    ho_rate_per_ue: float = 0.      # handovers per UE per slot
    ho_success_rate: float = 0.     # fraction of attempted HOs that succeed
    ho_failure_rate: float = 0.     # fraction of attempted HOs that fail
    ping_pong_rate: float = 0.      # fraction of HOs that are ping-pong
    policy_ho_fraction: float = 0.  # fraction of HOs triggered proactively by RL policy switch-offs
    mean_cit_per_ue: float = 0.     # mean Cell-in-Trouble (CIT) slots per UE
    mean_nactive_bs: float = 0.     # mean number of active BSs per slot
    cov_violation_rate: float = 0.  # fraction of slots with coverage below target


def run_episode(env_inst, policy, seed=None, deterministic=True):
    """
    Roll out one full episode (12 hours / 12 steps).
    Returns aggregated EpisodeMetrics reflecting the policy's performance.
    """
    obs, info = env_inst.reset(seed=seed)
    done = False
    p_sum = cov_sum = 0.            # power and coverage accumulators
    n_ho = n_hs = n_hf = n_pp = n_pol = 0  # HO event counters
    n_active_sum = n_viol = n_steps = 0     # activity and violation counters

    while not done:
        # deterministic=True ensures the RL agent acts optimally (argmax) rather than exploring
        action, _, _ = policy.act(obs, deterministic=deterministic)
        obs, reward, term, trunc, info = env_inst.step(action)
        done = term or trunc

        p_sum += info['powerw']                                  # accumulate power
        cov_sum += info['coverage']                              # accumulate coverage
        if info['coverage'] < TARGET_COVERAGE:
            n_viol += 1                                          # count coverage-violation slots
        n_active_sum += info['nactivebs']                        # count active BSs
        n_steps += 1

        ev = info['hoevents']
        n_ho += ev.get('nhandovers', 0)
        n_hs += ev.get('nhosuccess', 0)
        n_hf += ev.get('nhofailure', 0)
        n_pp += ev.get('nhopingpong', 0)
        n_pol += ev.get('nho_policy', 0)

    n_ue = info['nue']                                           # UE count at episode end
    cit = float(env_inst.hostate.cit_slots.mean()) if env_inst.hostate else 0.  # mean CIT per UE
    mp = p_sum / max(n_steps, 1)                                 # mean power

    return EpisodeMetrics(
        mean_power=mp,
        mean_cov=cov_sum / max(n_steps, 1),
        energy_saving_pct=(1. - mp / env_inst.max_power) * 100.,
        ho_rate_per_ue=n_ho / max(n_ue * n_steps, 1),
        ho_success_rate=n_hs / max(n_ho, 1),
        ho_failure_rate=n_hf / max(n_ho, 1),
        ping_pong_rate=n_pp / max(n_ho, 1),
        policy_ho_fraction=n_pol / max(n_ho, 1),
        mean_cit_per_ue=cit,
        mean_nactive_bs=n_active_sum / max(n_steps, 1),
        cov_violation_rate=n_viol / max(n_steps, 1),
    )


def evaluate_all_policies(n_episodes=5, base_seed=4321):
    """
    Evaluate all policies (RL + baselines) over n_episodes and return a summary DataFrame.
    Prints a detailed thesis-friendly configuration breakdown before running.
    """
    eval_env = HetNetEnergyEnv(BS_XY, BS_CT, USERCOUNTS, use_trust=True, t_on=T_ON_SLOTS)
    res = {}

    pols = {'RL-PPO': trained_model}
    pols.update(baseline_policies)

    hours_per_ep = eval_env.nslots # typically 12 hours (10:00 to 21:00)

    print(f"=== Thesis Evaluation Configuration ===")
    print(f"Timeframe snippet: Diurnal cycle from 10:00 to 21:00 (Total: {hours_per_ep} hours/steps per episode)")
    print(f"Episodes evaluated: {n_episodes} (Total simulated hours per policy: {n_episodes * hours_per_ep})")
    print(f"Fast Sim Steps (mobility/physics updates per hour): 60")
    print(f"Target Coverage: {TARGET_COVERAGE * 100}%\n")

    print("=== Policy Comparison Logic ===")
    print("- RL-PPO    : Proactive, predictive switch-off balancing energy, coverage, and handovers.")
    print("- Always-On : Traditional network baseline (upper bound for energy, lower bound for coverage violations).")
    print("- Random    : Unstructured toggling (baseline for instability/ping-pongs).")
    print("- Threshold : Reactive load-based heuristic (switches off only after consecutive low-load slots).")
    print("- Greedy    : Optimization heuristic (activates BSs sequentially based on highest TRUST scores).")
    print("---------------------------------------\n")

    for name, policy in pols.items():
        eps = [run_episode(eval_env, policy, seed=base_seed + k).__dict__ for k in range(n_episodes)]
        res[name] = pd.DataFrame(eps).mean().to_dict()

    df = pd.DataFrame(res).T
    print('\n=== Policy Evaluation Summary ===')
    print(df[['mean_power', 'mean_cov', 'energy_saving_pct',
               'ho_rate_per_ue', 'ho_failure_rate', 'ping_pong_rate',
               'mean_nactive_bs', 'cov_violation_rate']].round(3).to_string())

    df.to_csv(f'{OUTPUT_DIR}/policy_evaluation.csv')
    print(f'\nSaved -> {OUTPUT_DIR}/policy_evaluation.csv')
    return df




In [ ]:
# Run evaluation across 3 full diurnal episodes (3 * 12 = 36 hours simulated per policy)
eval_df = evaluate_all_policies(n_episodes=3)


In [ ]:
eval_df

## Cell 15 — Topology Preview

In [ ]:
# Cell 15 — Visualisation: 21-BS 4-tier HetNet topology with coverage circles

fig, ax = plt.subplots(figsize=(10, 10))  # Slightly larger for better thesis readability

for ct, name in enumerate(CELLTYPE_NAMES):
    mask = BS_CT == ct
    if not mask.any():
        continue

    # Plot BS positions with high z-order to stay above coverage circles
    ax.scatter(BS_XY[mask, 0], BS_XY[mask, 1],
               c=CT_COLORS[ct], marker=CT_MARKERS[ct],
               s=CT_SIZES[ct], edgecolors='black', linewidths=1.2,
               label=name, zorder=10 + ct)

    for j in np.where(mask)[0]:
        # Coverage circle with distinct edge for academic clarity
        c = plt.Circle((BS_XY[j, 0], BS_XY[j, 1]), MAX_DIST_M[ct],
                       facecolor=CT_COLORS[ct], edgecolor=CT_COLORS[ct],
                       alpha=0.08, linewidth=1.5, zorder=2 + ct)
        ax.add_patch(c)

        # Add labels with a semi-transparent white box for readability against the grid
        ax.annotate(bs_df.iloc[j]['label'], (BS_XY[j, 0], BS_XY[j, 1] + 15),
                    fontsize=8, ha='center', va='bottom', zorder=20,
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7))

# Restrict limits exactly to the area to handle massive Macro coverage circles safely
ax.set_xlim(-50, AREA_M + 50)
ax.set_ylim(-50, AREA_M + 50)
ax.set_aspect('equal')

# Academic styling
ax.set_xlabel('X Coordinate (m)', fontsize=12)
ax.set_ylabel('Y Coordinate (m)', fontsize=12)
ax.set_title('HetNet Topology — 21 BS, 4 Tiers', fontsize=14, pad=15)
ax.legend(loc='upper right', framealpha=0.9, title="Base Station Tiers")
ax.grid(True, linestyle='--', alpha=0.5, zorder=1)

# High DPI for thesis print quality
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/topology_preview.png', dpi=300, bbox_inches='tight')
plt.show()
print('Topology preview saved at 300 DPI for publication.')

## Cell 16 — BS Activation Heatmap

In [ ]:
# Cell 16 — Visualisation: BS Activation Heatmap and Performance Dynamics

from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt
import numpy as np

# 1. Roll out a single deterministic evaluation episode using the trained model
vis_env = HetNetEnergyEnv(BS_XY, BS_CT, USERCOUNTS, use_trust=True, t_on=T_ON_SLOTS)
obs, _ = vis_env.reset(seed=2024)
done = False

while not done:
    # Use the trained RL agent optimally (deterministic=True)
    action, _, _ = trained_model.act(obs, deterministic=True)
    obs, _, terminated, truncated, _ = vis_env.step(action)
    done = terminated or truncated

history = vis_env.history

# 2. Extract state matrix
n_t = len(history)
n_bs = vis_env.n_bs
hmap = np.zeros((n_t, n_bs), dtype=np.int32)
for t, step in enumerate(history):
    hmap[t] = step['bs_state']  # record state machine value per slot

# 3. Create Thesis-Quality Plot
cmap = ListedColormap(['#E53935', '#FDD835', '#43A047'])  # Red=OFF, Yellow=WARMING, Green=ON

# Use gridspec to tightly align the two plots
fig, axes = plt.subplots(2, 1, figsize=(12, 10), gridspec_kw={'height_ratios': [3, 1.2], 'hspace': 0.3})

# --- TOP PLOT: Base Station Activation Heatmap ---
ax1 = axes[0]
im = ax1.imshow(hmap.T, aspect='auto', cmap=cmap, vmin=0, vmax=2, interpolation='nearest')

ax1.set_yticks(range(n_bs))
ax1.set_yticklabels(bs_df['label'].tolist(), fontsize=9)
ax1.set_xticks(range(n_t))
ax1.set_xticklabels([f'{h}:00' for h in SLOT_HOURS[:n_t]], rotation=0, fontsize=10)

ax1.set_xlabel('Simulation Time (Diurnal Cycle)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Base Station Identifier', fontsize=11, fontweight='bold')
ax1.set_title('RL Agent Policy: Hourly Base Station State Transitions', fontsize=14, pad=15)

# Add a clean colorbar
cbar = plt.colorbar(im, ax=ax1, ticks=[0, 1, 2], fraction=0.03, pad=0.02)
cbar.ax.set_yticklabels(['Sleep (OFF)', 'Warming Up', 'Active (ON)'], fontsize=10)

# --- BOTTOM PLOT: Coverage vs Traffic Load ---
ax2 = axes[1]
covs = [s['coverage'] for s in history]
nues = [len(s['ue_xy']) for s in history]

# Coverage Line
line1 = ax2.plot(range(n_t), covs, marker='o', markersize=6, linewidth=2, color='#1565C0', label='Actual Coverage')

# Target Coverage Threshold Line
line2 = ax2.axhline(y=TARGET_COVERAGE, color='red', linestyle='--', linewidth=1.5, alpha=0.8, label=f'Target ({TARGET_COVERAGE*100}%)')

ax2.set_ylim(0, 1.1)
ax2.set_ylabel('Coverage Fraction', color='#1565C0', fontsize=11, fontweight='bold')
ax2.tick_params(axis='y', labelcolor='#1565C0')

# UE Count Bars (Twin Axis)
ax2b = ax2.twinx()
bars = ax2b.bar(range(n_t), nues, width=0.4, alpha=0.25, color='#E65100', label='UE Traffic Load')
ax2b.set_ylabel('Active Users (UEs)', color='#E65100', fontsize=11, fontweight='bold')
ax2b.tick_params(axis='y', labelcolor='#E65100')

ax2.set_xticks(range(n_t))
ax2.set_xticklabels([f'{h}:00' for h in SLOT_HOURS[:n_t]], rotation=0, fontsize=10)
ax2.set_title('Network Performance vs. Traffic Load', fontsize=13, pad=10)
ax2.grid(True, linestyle='--', alpha=0.4)

# Combine legends from both axes
lines, labels = ax2.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.4))

plt.savefig(f'{OUTPUT_DIR}/bs_activation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print('BS activation heatmap saved at 300 DPI for publication.')

## Cell 17 — BS ON/OFF Map Snapshots

In [ ]:
# Cell 17 — Visualisation: BS ON/OFF map snapshots at beginning, middle, and end of episode
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Choose three meaningful slots: Start (10:00), Peak (15:00), End (21:00)
snap_slots = [0, 5, 11]
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# Create a background grid for the SINR heatmap
grid_res = 100
x_grid = np.linspace(0, AREA_M, grid_res)
y_grid = np.linspace(0, AREA_M, grid_res)
xx, yy = np.meshgrid(x_grid, y_grid)
grid_xy = np.column_stack((xx.ravel(), yy.ravel()))

for ax, t_idx in zip(axes, snap_slots):
    t_idx = min(t_idx, len(history) - 1)
    step = history[t_idx]

    am = step['activemask']             # active BS mask at this slot
    bsst = step['bs_state']             # BS state machine values
    ue_xy_snap = step['ue_xy']          # UE positions at this slot
    serv = step['serving']              # serving BS per UE

    # 1. Background SINR Heatmap
    # Calculate channel gain for the grid points to generate the heatmap
    cg_grid = channel_gain_linear(grid_xy, BS_XY, BS_CT)
    sinr_grid = sinr_db_matrix(cg_grid, am, BS_CT)
    best_sinr_grid = np.max(np.where(am[None, :] == 1, sinr_grid, -120.), axis=1)
    best_sinr_grid = best_sinr_grid.reshape(grid_res, grid_res)

    # Plot contour: blue/green is good signal, yellow/red is poor/dead zone
    cntr = ax.contourf(xx, yy, best_sinr_grid, levels=np.linspace(-10, 30, 20),
                       cmap='viridis_r', alpha=0.3, zorder=0)

    # 2. Simulation Boundary
    rect = plt.Rectangle((0, 0), AREA_M, AREA_M, fill=False, edgecolor='black',
                         linewidth=2, linestyle='--', zorder=1)
    ax.add_patch(rect)

    # 3. User Equipment (UE) & Connection Lines
    ue_cols = []
    for i in range(len(ue_xy_snap)):
        j = serv[i]
        is_served = (j >= 0 and am[j] == 1)
        col = CT_COLORS[BS_CT[j]] if is_served else '#FF0000' # Red if dropped/dead zone
        ue_cols.append(col)

        # Draw faint line from UE to serving BS
        if is_served:
            ax.plot([ue_xy_snap[i, 0], BS_XY[j, 0]],
                    [ue_xy_snap[i, 1], BS_XY[j, 1]],
                    color=col, linewidth=0.2, alpha=0.4, zorder=2)

    # Scatter UEs
    ax.scatter(ue_xy_snap[:, 0], ue_xy_snap[:, 1], c=ue_cols, s=8, alpha=0.9, zorder=3, edgecolors='none')

    # 4. Base Stations (Nodes)
    for j in range(vis_env.n_bs):
        ct = BS_CT[j]; st = bsst[j]
        if st == HetNetEnergyEnv.BS_ON:
            ec, fc, al = 'black', CT_COLORS[ct], 1.
            # Coverage circles for active BSs
            c = plt.Circle((BS_XY[j, 0], BS_XY[j, 1]), MAX_DIST_M[ct],
                           facecolor=CT_COLORS[ct], edgecolor='black', alpha=0.08, linewidth=0.5, zorder=1)
            ax.add_patch(c)
        elif st == HetNetEnergyEnv.BS_WARMING:
            ec, fc, al = 'orange', '#FDD835', 0.9
        else:
            ec, fc, al = '#555555', '#CCCCCC', 0.5  # OFF

        ax.scatter(BS_XY[j, 0], BS_XY[j, 1], marker=CT_MARKERS[ct], s=CT_SIZES[ct]*1.5,
                   c=fc, edgecolors=ec, linewidths=1.5, alpha=al, zorder=6)

        # Labels for BS
        ax.annotate(bs_df.iloc[j]['label'], (BS_XY[j, 0], BS_XY[j, 1] + 15),
                    fontsize=7, ha='center', va='bottom', zorder=7,
                    bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.6))

    # 5. Formatting & Titles
    hl = SLOT_HOURS[t_idx] if t_idx < len(SLOT_HOURS) else t_idx
    ax.set_xlim(-50, AREA_M + 50)
    ax.set_ylim(-50, AREA_M + 50)
    ax.set_aspect('equal')

    # Detailed sub-titles showing the exact tradeoff
    title_str = (f"Time: {hl}:00 | Traffic: {len(ue_xy_snap)} UEs\n"
                 f"Active BSs: {int(am.sum())}/21 | Power: {step['power']:.0f} W\n"
                 f"Coverage: {step['coverage']*100:.1f}%")
    ax.set_title(title_str, fontsize=11, pad=10, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle=':')
    ax.set_xlabel('X Coordinate (m)')
    if ax == axes[0]:
        ax.set_ylabel('Y Coordinate (m)')

# 6. Global Legend
patches = [mpatches.Patch(color=CT_COLORS[ct], label=CELLTYPE_NAMES[ct]) for ct in range(4)]
patches += [mpatches.Patch(color='#CCCCCC', label='BS OFF'),
            mpatches.Patch(color='#FDD835', label='BS WARMING'),
            mpatches.Patch(color='#FF0000', label='Unserved UE (Dropped)')]

fig.legend(handles=patches, loc='lower center', ncol=7, fontsize=10, bbox_to_anchor=(0.5, 0.02))
plt.suptitle('Spatial Analysis of RL Agent Decisions: SINR Heatmap, Load, and Topology',
             fontsize=16, fontweight='bold', y=0.98)

# Add colorbar for SINR
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
cbar = fig.colorbar(cntr, cax=cbar_ax)
cbar.set_label('Best Available SINR (dB)', rotation=270, labelpad=15, fontweight='bold')

plt.subplots_adjust(bottom=0.15, wspace=0.1, right=0.9)
plt.savefig(f'{OUTPUT_DIR}/bs_snapshot_maps_thesis.png', dpi=300, bbox_inches='tight')
plt.show()
print('Comprehensive Spatial snapshots saved at 300 DPI.')

## Cell 18 — Training Curves

In [ ]:
# Cell 18 — Visualisation: reward component training curves (smoothed)

import matplotlib.pyplot as plt
import numpy as np

# Ensure you have the logs from the primary seed (e.g., 42)
primary_seed = TRAIN_SEEDS[0]
logs = all_results[primary_seed]['logs']

def smooth(arr, w=500):
    """
    Uniform 1D smoothing via convolution with edge-padding.
    Prevents the smoothed line from shrinking relative to the raw data.
    """
    if len(arr) < w:
        return np.array(arr)
    padded = np.pad(arr, (w//2, w-1-w//2), mode='edge')
    return np.convolve(padded, np.ones(w)/w, mode='valid')

# Define exactly what each plot represents for the thesis
pairs = [
    ('rewards', 'Total Cumulative Reward', 'Total Reward'),
    ('power', 'Network Power Consumption', 'Power (W)'),
    ('cov', 'Coverage Satisfaction Fraction', 'Coverage (0-1)'),
    ('r_E', 'Energy Reward Component ($r_E$)', 'Reward'),
    ('r_C', 'Coverage Reward Component ($r_C$)', 'Reward'),
    ('r_H', 'Handover Penalty Component ($r_H$)', 'Penalty')
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, (key, title, ylabel) in zip(axes.flatten(), pairs):
    raw = np.array(logs[key])

    # Plot raw noisy data
    ax.plot(raw, alpha=0.3, color='#4682B4', linewidth=0.8, label='Raw Epoch Data')

    # Plot smoothed trend line
    if len(raw) >= 500:
        sm = smooth(raw, 500)
        ax.plot(sm, color='#000080', linewidth=2.5, label='Moving Average (w=500)')

    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('Environment Steps', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(fontsize=9, loc='best')

# Thesis-level master title
plt.suptitle(f'PPO Agent Convergence Metrics Over Training Timeline (Seed: {primary_seed})',
             fontsize=16, fontweight='bold', y=0.98)

# Adjust layout and save at publication quality
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(f'{OUTPUT_DIR}/training_curves_thesis.png', dpi=300, bbox_inches='tight')
plt.show()

print('Publication-quality training curves saved.')

## Cell 19 — UE Trajectories + HO Events

In [ ]:
# Cell 19 — Micro-Simulation Visualisation: 60-Second Intra-Hour Dynamics (20:00 → 21:00)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

eval_seed = 2024
target_slot = 10  # Evaluate transition from 20:00 to 21:00
FAST_SIM_STEPS = 60
FAST_DT = 1.0

# Create figure with extra bottom space for the tabular legend
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
plt.subplots_adjust(bottom=0.25)

titles = ['RL-PPO: Policy & Mobility Handovers (20:00 → 21:00)',
          'Always-On: Pure Mobility Handovers (20:00 → 21:00)']
pols2 = [trained_model, baseline_policies['Always-On']]

grid_res = 100
x_grid, y_grid = np.meshgrid(np.linspace(0, AREA_M, grid_res), np.linspace(0, AREA_M, grid_res))
grid_xy = np.column_stack((x_grid.ravel(), y_grid.ravel()))
cg_grid = channel_gain_linear(grid_xy, BS_XY, BS_CT)

for ax, pol, title in zip(axes, pols2, titles):
    ve = HetNetEnergyEnv(BS_XY, BS_CT, USERCOUNTS, use_trust=True, t_on=T_ON_SLOTS)
    obs, _ = ve.reset(seed=eval_seed)

    # Fast-forward to 20:00
    for _ in range(target_slot):
        action, _, _ = pol.act(obs, deterministic=True)
        obs, _, _, _, _ = ve.step(action)

    # --- INTERCEPT THE 20:00 -> 21:00 TRANSITION ---
    action, _, _ = pol.act(obs, deterministic=True)
    ve.activemask = ve._apply_sm(np.asarray(action, dtype=np.int32))
    ve.slot += 1
    n_ue = ve.uecounts[min(ve.slot, ve.nslots - 1)]

    # Background SINR
    sinr_grid = sinr_db_matrix(cg_grid, ve.activemask, BS_CT)
    best_sinr = np.max(np.where(ve.activemask[None, :] == 1, sinr_grid, -120.), axis=1).reshape(grid_res, grid_res)
    cntr = ax.contourf(x_grid, y_grid, best_sinr, levels=np.linspace(-10, 30, 20), cmap='viridis_r', alpha=0.3, zorder=0)

    # Plot Base Stations
    for j in range(ve.n_bs):
        ct = BS_CT[j]
        is_active = ve.activemask[j] == 1
        color = CT_COLORS[ct] if is_active else '#CCCCCC'
        edge = 'black' if is_active else '#555555'
        ax.scatter(BS_XY[j, 0], BS_XY[j, 1], marker=CT_MARKERS[ct], s=CT_SIZES[ct]*1.5,
                   c=color, edgecolors=edge, linewidths=1.2, zorder=5, alpha=0.9)
        if is_active:
            ax.add_patch(plt.Circle((BS_XY[j, 0], BS_XY[j, 1]), MAX_DIST_M[ct], color=color, alpha=0.05, zorder=1))
        # Add tiny unobtrusive BS labels
        ax.annotate(str(j), (BS_XY[j, 0], BS_XY[j, 1]-15), fontsize=8, color='black', ha='center')

    # --- MANUAL MICRO-SIMULATION ---
    ue_trajs = {i: [] for i in range(n_ue)}
    ho_points = []

    cg = channel_gain_linear(ve.mob.pos[:n_ue], ve.bs_xy, ve.bs_ct, ve.rng)
    _, _, _, _, sinr_mat = greedy_association(cg, ve.activemask, ve.bs_ct)
    ve.hostate.update(sinr_mat, ve.activemask)
    prev_serv = ve.hostate.serving.copy()[:n_ue]

    for sec in range(FAST_SIM_STEPS):
        full_ue_xy = ve.mob.step(dt=FAST_DT)
        ve.ue_xy = full_ue_xy[:n_ue].copy()

        cg = channel_gain_linear(ve.ue_xy, ve.bs_xy, ve.bs_ct, ve.rng)
        _, _, _, _, sinr_mat = greedy_association(cg, ve.activemask, ve.bs_ct)
        ve.hostate.update(sinr_mat, ve.activemask)
        curr_serv = ve.hostate.serving.copy()[:n_ue]

        for i in range(n_ue):
            ue_trajs[i].append(ve.ue_xy[i].copy())
            if curr_serv[i] != prev_serv[i] and prev_serv[i] >= 0:
                ho_points.append({
                    'ue_id': i, 'x': ve.ue_xy[i, 0], 'y': ve.ue_xy[i, 1],
                    'sec': sec + 1, 'from': prev_serv[i], 'to': curr_serv[i]
                })
        prev_serv = curr_serv.copy()

    # --- PLOTTING TRAJECTORIES & HANDOVERS ---
    # Limit to 15 handovers to prevent visual chaos
    np.random.seed(42)  # Fixed for consistent display
    if len(ho_points) > 15:
        disp_ho_points = np.random.choice(ho_points, 15, replace=False)
    else:
        disp_ho_points = ho_points

    disp_ue_ids = set([hp['ue_id'] for hp in disp_ho_points])

    # Plot non-HO UEs faintly
    for i in range(n_ue):
        if i not in disp_ue_ids:
            traj = np.array(ue_trajs[i])
            ax.plot(traj[:, 0], traj[:, 1], alpha=0.1, linewidth=0.5, color='grey', zorder=2)

    # Plot HO UEs boldly, assign numeric IDs instead of full text
    ho_details_list = []
    for idx, hp in enumerate(disp_ho_points):
        i = hp['ue_id']
        traj = np.array(ue_trajs[i])
        ax.plot(traj[:, 0], traj[:, 1], alpha=0.9, linewidth=1.5, color='black', zorder=3)
        ax.scatter(hp['x'], hp['y'], s=80, color='red', marker='X', zorder=6, edgecolors='white')

        # Plot just the clean numeric ID next to the 'X'
        ax.annotate(str(idx + 1), (hp['x'] + 10, hp['y'] + 10),
                    fontsize=9, color='darkred', weight='bold', zorder=7)

        # Save details for the legend table
        ho_details_list.append(f"[{idx + 1}]: Sec {hp['sec']}, BS {hp['from']} → {hp['to']}")

    # Academic Formatting
    ax.set_xlim(-50, AREA_M + 50); ax.set_ylim(-50, AREA_M + 50); ax.set_aspect('equal')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel('X Coordinate (m)'); ax.set_ylabel('Y Coordinate (m)')
    ax.grid(True, linestyle=':', alpha=0.6)

    # Simple top-left box
    props = dict(boxstyle='square', facecolor='white', alpha=0.9, edgecolor='black')
    ax.text(0.02, 0.98, f"Hour: 21:00\nActive BSs: {int(ve.activemask.sum())}/21\nTotal HOs: {len(ho_points)}",
            transform=ax.transAxes, fontsize=11, verticalalignment='top', bbox=props, zorder=10)

    # Print tabulated details cleanly below the plot
    detail_str = "Selected Handover Details:\n"
    # Format into 3 columns
    for chunk in [ho_details_list[i:i+3] for i in range(0, len(ho_details_list), 3)]:
        detail_str += "    ".join([s.ljust(25) for s in chunk]) + "\n"

    ax.text(0.0, -0.15, detail_str, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', fontfamily='monospace')

plt.suptitle('60-Second Micro-Simulation: Clean Trajectories with Tabulated Handover Events',
             fontsize=16, fontweight='bold', y=0.96)
plt.savefig(f'{OUTPUT_DIR}/micro_sim_ho_clean.png', dpi=300, bbox_inches='tight')
plt.show()
print('Cleaned micro-simulation tracking plot saved.')

## Cell 20 — Bar Charts vs Baselines

In [ ]:
# Cell 20 — Visualisation: Multi-Objective Policy Benchmarking

import numpy as np
import matplotlib.pyplot as plt

if 'eval_df' not in dir():
    print("Running evaluation suite. This may take a moment...")
    eval_df = evaluate_all_policies(n_episodes=3)

# Setup a 1x4 grid to include the Pie Chart
fig, axes = plt.subplots(1, 4, figsize=(20, 6))

pnames = eval_df.index.tolist()

# Highlight RL-PPO in Orange, Baselines in Steel Blue
colors = ['#FF8C00' if name == 'RL-PPO' else '#4682B4' for name in pnames]

metrics = [
    ('mean_power', 'Network Power Consumption (W)'),
    ('mean_cov', 'Coverage Satisfaction (Fraction)'),
    ('ho_rate_per_ue', 'Handovers per UE (Rate)')
]

# --- PLOT 1, 2, 3: Bar Charts ---
for i, (metric, title) in enumerate(metrics):
    ax = axes[i]
    vals = eval_df[metric].values.astype(float)
    bars = ax.bar(pnames, vals, color=colors, edgecolor='black', linewidth=1.2, alpha=0.85)

    ax.set_ylabel(title, fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=13, pad=10)
    ax.set_xticks(range(len(pnames)))
    ax.set_xticklabels(pnames, rotation=25, ha='right', fontsize=11)
    ax.grid(axis='y', linestyle='--', alpha=0.5, zorder=0)

    # Add data labels on top of the bars
    for bar, v in zip(bars, vals):
        # Format dynamically: integers for power, 3 decimals for fractions/rates
        fmt = f'{v:.0f}' if metric == 'mean_power' else f'{v:.3f}'
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + (vals.max() * 0.02),
                fmt, ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Add Target Line for Coverage Plot
    if metric == 'mean_cov':
        ax.axhline(y=TARGET_COVERAGE, color='red', linestyle='--', linewidth=2,
                   label=f'Target ({TARGET_COVERAGE*100}%)')
        ax.set_ylim(0, 1.1)
        ax.legend(loc='lower right', fontsize=9)

# --- PLOT 4: RL-PPO Handover Decomposition Pie Chart ---
ax_pie = axes[3]
if 'RL-PPO' in eval_df.index:
    rl_data = eval_df.loc['RL-PPO']

    # We plot Success vs Failure vs Ping-Pong
    # Note: Success + Failure = 1.0. Ping-pong is a subset of success.
    # To plot properly: Failure, Ping-Pong, and (Success minus Ping-Pong)
    pp_rate = float(rl_data['ping_pong_rate'])
    fail_rate = float(rl_data['ho_failure_rate'])
    pure_success = max(0, 1.0 - fail_rate - pp_rate)  # Ensure non-negative

    labels = ['Failed HOs\n(Target Offline)', 'Ping-Pong HOs\n(Oscillation)', 'Successful HOs\n(Stable)']
    sizes = [fail_rate, pp_rate, pure_success]
    pie_colors = ['#c0392b', '#f39c12', '#27ae60']
    explode = (0.1, 0, 0)  # Emphasize the failure rate

    # Only plot segments that actually exist
    filt_labels = [l for l, s in zip(labels, sizes) if s > 0]
    filt_sizes = [s for s in sizes if s > 0]
    filt_colors = [c for c, s in zip(pie_colors, sizes) if s > 0]
    filt_explode = [e for e, s in zip(explode, sizes) if s > 0]

    ax_pie.pie(filt_sizes, explode=filt_explode, labels=filt_labels, colors=filt_colors,
               autopct='%1.1f%%', shadow=False, startangle=140, textprops={'fontsize': 10, 'weight': 'bold'})
    ax_pie.set_title("RL-PPO: Handover Execution Quality", fontsize=13, pad=10)

plt.suptitle('Comprehensive Multi-Objective Policy Benchmarking vs Baseline Heuristics',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()

plt.savefig(f'{OUTPUT_DIR}/policy_comparison_benchmarks.png', dpi=300, bbox_inches='tight')
plt.show()

print('Intuitive Multi-Objective Bar & Pie Charts saved at 300 DPI.')

## Cell 21 — HO-Energy Pareto Frontier (W_HO sweep)

Sweep the handover penalty weight `W_HO` from 0 to 5 to trace the Pareto frontier
between energy consumption and handover rate.  
Set `N_SWEEP = 0` to skip (each point trains `SWEEP_STEPS` PPO steps).


In [ ]:
# Cell 21 — Visualisation: HO-Energy Pareto frontier by sweeping W_HO

import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm

print("Starting Pareto Frontier Sweep...")
print("Training multiple mini-models to map the trade-off between Energy and Handovers.\n")

N_SWEEP = 5                                  # Number of W_HO penalty values to test
SWEEP_STEPS = 5000                           # Increased from 2500 to give models time to form a distinct policy
W_HO_VALUES = np.linspace(0.0, 5.0, N_SWEEP) if N_SWEEP > 0 else []

pareto_power = []
pareto_ho = []
pareto_labels = []

# 1. Train and Evaluate Models across the Sweep
for idx, w_ho in enumerate(W_HO_VALUES):
    print(f"[{idx+1}/{N_SWEEP}] Training model with Handover Penalty W_HO = {w_ho:.1f}")

    # Instantiate a fresh environment and fresh model for this specific weight
    sw_env = HetNetEnergyEnv(BS_XY, BS_CT, USERCOUNTS, use_trust=True, t_on=T_ON_SLOTS, w_ho=float(w_ho))
    sw_model = ActorCritic(n_bs=sw_env.n_bs, bs_feats=sw_env.n_bs_feats).to(DEVICE)
    sw_opt = torch.optim.Adam(sw_model.parameters(), lr=PPO_LR)

    obs, _ = sw_env.reset(seed=42)
    rollout = []

    # Fast Training Loop for the Sweep
    for si in tqdm(range(SWEEP_STEPS), desc="Training", leave=False):
        action, logp, val = sw_model.act(obs)
        nobs, reward, term, trunc, info = sw_env.step(action)
        rollout.append((obs, action.astype(np.float32), logp, reward, val, float(term)))
        obs = nobs

        if len(rollout) >= PPO_ROLLOUT_LEN:
            ppo_update(sw_model, sw_opt, rollout, si, SWEEP_STEPS)
            rollout = []

        if term:
            obs, _ = sw_env.reset()

    # Evaluate the newly trained mini-model
    sw_model.eval()
    m = run_episode(sw_env, sw_model, seed=999)

    pareto_power.append(m.mean_power)
    pareto_ho.append(m.ho_rate_per_ue)
    pareto_labels.append(f'W_HO = {w_ho:.1f}')

    print(f" -> Result: Power = {m.mean_power:.0f} W | HO Rate = {m.ho_rate_per_ue:.4f} per UE\n")

# 2. Plot the Pareto Frontier
if pareto_power:
    fig, ax = plt.subplots(figsize=(9, 6))

    # Scatter plot mapped to a colormap (Blue = Low penalty, Red = High penalty)
    sc = ax.scatter(pareto_ho, pareto_power,
                    c=W_HO_VALUES, cmap='coolwarm',
                    s=150, edgecolors='black', linewidths=1.5, zorder=5)

    # Connect the dots to show the frontier curve
    ax.plot(pareto_ho, pareto_power, linestyle='--', color='gray', alpha=0.6, linewidth=2, zorder=4)

    # Annotate each point with its W_HO value
    for i, lbl in enumerate(pareto_labels):
        ax.annotate(lbl, (pareto_ho[i], pareto_power[i]),
                    textcoords='offset points', xytext=(10, 5),
                    fontsize=10, fontweight='bold')

    # Academic formatting
    ax.set_xlabel('Handover Rate (Events per UE per slot)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean Network Power Consumption (W)', fontsize=12, fontweight='bold')
    ax.set_title('Multi-Objective Pareto Frontier: Energy vs. Handover Trade-off', fontsize=14, pad=15)

    # Add an arrow pointing to the "Ideal" but unachievable zone
    ax.annotate('Ideal Optimization\n(Low Power, Low HOs)',
                xy=(min(pareto_ho), min(pareto_power)),
                xytext=(min(pareto_ho)*0.8, min(pareto_power)*0.8),
                arrowprops=dict(facecolor='green', shrink=0.05, alpha=0.5),
                fontsize=11, color='green', fontweight='bold', ha='center')

    # Colorbar to represent the W_HO spectrum
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('Handover Penalty Weight ($W_{HO}$)', rotation=270, labelpad=20, fontweight='bold')

    ax.grid(True, linestyle=':', alpha=0.7)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/pareto_frontier_thesis.png', dpi=300, bbox_inches='tight')
    plt.show()
    print('Pareto frontier saved at 300 DPI.')
else:
    print('N_SWEEP=0 — skipped.')

## Cell 22 — Ablation Study

Five ablation configurations isolate the contribution of each design choice:

| Config | What is removed |
|--------|----------------|
| `full` | Complete model (all features enabled) |
| `no_trust` | TRUST feature removed from observations |
| `no_handover_reward` | W_HO = W_PP = W_CI = 0 |
| `no_warmup` | BS warm-up delay set to 0 slots |
| `cov_target_0.98` | Coverage target relaxed to 98 % |


In [ ]:
# Cell 22 — Ablation study runner (short pilot budget — increase for publication quality)

import pandas as pd
import torch

ABLATION_STEPS = 3000  # Increased slightly so the agent actually learns the ablation difference

print("Starting Ablation Study. Evaluating the impact of removing key architecture components...")

# The dictionary defines what we override in each ablation test
ablation_configs = {
    'Full Architecture': dict(use_trust=True, t_on=T_ON_SLOTS, w_ho=W_HO, w_pp=W_PING_PONG, w_ci=W_CIT, target_cov=TARGET_COVERAGE),
    'No TRUST Logic':    dict(use_trust=False, t_on=T_ON_SLOTS, w_ho=W_HO, w_pp=W_PING_PONG, w_ci=W_CIT, target_cov=TARGET_COVERAGE),
    'No HO Penalty':     dict(use_trust=True, t_on=T_ON_SLOTS, w_ho=0.0, w_pp=0.0, w_ci=0.0, target_cov=TARGET_COVERAGE),
    'No Warmup Delay':   dict(use_trust=True, t_on=0, w_ho=W_HO, w_pp=W_PING_PONG, w_ci=W_CIT, target_cov=TARGET_COVERAGE),
    'Strict 100% Target':dict(use_trust=True, t_on=T_ON_SLOTS, w_ho=W_HO, w_pp=W_PING_PONG, w_ci=W_CIT, target_cov=1.0) # Assuming baseline target is now 0.98
}

ablation_results = {}

for abl_name, cfg in ablation_configs.items():
    print(f'\n--- Training Ablation Config: {abl_name} ---')

    # Temporarily override the global target coverage for this specific run
    # (Since TARGET_COVERAGE is hardcoded globally in your notebook)
    global TARGET_COVERAGE
    original_target = TARGET_COVERAGE
    TARGET_COVERAGE = cfg['target_cov']

    try:
        # Initialize environment with specific ablation parameters
        abl_env = HetNetEnergyEnv(BS_XY, BS_CT, USERCOUNTS,
                                  use_trust=cfg['use_trust'], t_on=cfg['t_on'],
                                  w_ho=cfg['w_ho'], w_pp=cfg['w_pp'], w_ci=cfg['w_ci'])

        abl_model = ActorCritic(n_bs=abl_env.n_bs, bs_feats=abl_env.n_bs_feats).to(DEVICE)
        abl_opt = torch.optim.Adam(abl_model.parameters(), lr=PPO_LR)
        obs, _ = abl_env.reset(seed=42)
        rollout = []

        # Train
        for si in range(ABLATION_STEPS):
            action, logp, val = abl_model.act(obs)
            nobs, reward, term, trunc, info = abl_env.step(action)
            rollout.append((obs, action.astype(np.float32), logp, reward, val, float(term)))
            obs = nobs

            if len(rollout) >= PPO_ROLLOUT_LEN:
                ppo_update(abl_model, abl_opt, rollout, si, ABLATION_STEPS)
                rollout = []
            if term:
                obs, _ = abl_env.reset()

        # Evaluate
        abl_model.eval()
        m = run_episode(abl_env, abl_model, seed=999)
        ablation_results[abl_name] = m.__dict__

        print(f' Power: {m.mean_power:.1f}W | Coverage: {m.mean_cov*100:.1f}% | '
              f'HO Rate: {m.ho_rate_per_ue:.4f} | Ping-Pong: {m.ping_pong_rate:.4f}')

    finally:
        # Guarantee the global target coverage is restored even if the loop crashes
        TARGET_COVERAGE = original_target


# Format and save the results into a clean thesis table
abl_df = pd.DataFrame(ablation_results).T
print('\n=== Ablation Results Summary ===')
print(abl_df[['mean_power', 'mean_cov', 'energy_saving_pct',
               'ho_rate_per_ue', 'ping_pong_rate', 'mean_cit_per_ue']].round(4).to_string())

abl_df.to_csv(f'{OUTPUT_DIR}/ablation_results_thesis.csv')
print(f'\nSaved -> {OUTPUT_DIR}/ablation_results_thesis.csv')

## Cell 23 — Ablation Radar Chart

In [ ]:
# Cell 23 — Visualisation: Ablation Radar Chart (Normalized, Outward = Better)

import numpy as np
import matplotlib.pyplot as plt

if 'abl_df' not in dir() or len(abl_df) == 0:
    print('Run Cell 22 first.'); raise SystemExit

metrics_radar = ['energy_saving_pct', 'mean_cov', 'ho_rate_per_ue',
                 'ping_pong_rate', 'ho_failure_rate', 'mean_cit_per_ue']

# Cleaner labels for academic presentation
labels_radar = ['Energy\nEfficiency', 'Coverage\nSatisfaction',
                'Handover\nStability', 'Ping-Pong\nPrevention',
                'Handover\nSuccess', 'CIT\nPrevention']

rad_df = abl_df[metrics_radar].copy().astype(float)

# Min-max normalise each metric to [0, 1]
for col in metrics_radar:
    mn, mx = rad_df[col].min(), rad_df[col].max()
    rad_df[col] = (rad_df[col] - mn) / (mx - mn) if mx > mn else 0.5

# Invert penalty metrics so that outward = better on the radar
for col in ['ho_rate_per_ue', 'ping_pong_rate', 'ho_failure_rate', 'mean_cit_per_ue']:
    rad_df[col] = 1.0 - rad_df[col]

N = len(metrics_radar)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close polygon

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

# Custom color mapping to highlight the 'Full Architecture'
colors_radar = []
for name in rad_df.index:
    if 'Full' in name or 'full' in name:
        colors_radar.append('#27ae60')  # Strong Green for the proposed full model
    else:
        # Muted colors for the ablated versions
        colors_radar.append(plt.cm.tab10(len(colors_radar) % 10))

for (name, row), col in zip(rad_df.iterrows(), colors_radar):
    vals = row.tolist() + [row.tolist()[0]]  # close polygon

    # Highlight the full model by making it thicker and bringing it to the front
    if 'Full' in name or 'full' in name:
        lw = 2.5
        alpha_fill = 0.2
        z = 10
    else:
        lw = 1.2
        alpha_fill = 0.05
        z = 5

    ax.plot(angles, vals, '-o', linewidth=lw, color=col, label=name, markersize=6, zorder=z)
    ax.fill(angles, vals, alpha=alpha_fill, color=col, zorder=z-1)

# Formatting
ax.set_thetagrids(np.degrees(angles[:-1]), labels_radar, fontsize=11, fontweight='bold')
ax.set_ylim(0, 1)

# Remove the confusing radial y-axis labels (0.2, 0.4, etc.)
ax.set_yticklabels([])

# Add clear concentric grid lines
ax.grid(True, color='gray', linestyle=':', linewidth=0.5, alpha=0.7)

ax.set_title('Ablation Study: Multi-Objective Performance\n(Outer Edge = Optimal Performance)',
             pad=30, fontsize=15, fontweight='bold')

# Beautiful legend placed outside the radar
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.05), fontsize=10,
          title="Model Configurations", title_fontsize=11, framealpha=0.9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ablation_radar_thesis.png', dpi=300, bbox_inches='tight')
plt.show()

print('Thesis-ready Ablation Radar chart saved at 300 DPI.')